# 05: Phương pháp nhóm: PP3 (tri thức X→Y từ WikiHow) + PP4 (RoBERTa+BiLSTM+CRF) + ablation + CV · lượt Kaggle **K2**

**Mục tiêu (H1–H3):** huấn luyện các phiên bản của mô hình nhóm, chọn epoch trên **dev**, đo đóng góp từng thành phần; rồi so **M3 với M0c** bằng CV 5 phần theo chủ đề trên lát **ngầm**.

| Mã | Biến thể (`variant`) | Ý nghĩa |
|---|---|---|
| M0 | `M0` | phân loại từng ô độc lập (cross-entropy), không ràng buộc |
| M0c | `M0` + Viterbi ràng buộc | **cùng mô hình M0**, chỉ thêm luật tồn tại khi giải mã → tách giá trị của *luật* |
| M3-no-pair | `CRF_NOPAIR` | CRF giải mã chung, huấn luyện −log p(y_vàng), không điểm cặp |
| M3-no-E | `CRF_NOE` | CRF + điểm cặp ψ **không** có log E(X,Y) |
| M2r | `CRF_RANDE` | CRF + ψ với log E **xáo trộn ngẫu nhiên** (kiểm tra “thêm đặc trưng” ≠ “tri thức hữu ích”) |
| **M3** | `CRF_FULL` | CRF + ψ + log E(X,Y) từ PP3 (**phương pháp nhóm đầy đủ**) |

**PP2:** trong giải mã chính xác, ràng buộc tồn tại đã bao hàm bộ lọc PP2. PP2 được đo riêng như **độ phủ cặp vàng khi dùng tồn tại dự đoán** (H1).

**Quy tắc khoá trước:**
- chọn epoch theo PAIR_F1 dev (hoà thì lấy dev loss thấp hơn);
- 3 seed;
- CV dùng số epoch = trung vị epoch tốt nhất trên dev của biến thể đó (không chỉnh trên fold kiểm tra);
- dự đoán **test được lưu kín, KHÔNG chấm** trong notebook này (chấm ở bước khoá K3).

### Cài đặt Kaggle
| Mục | Giá trị |
|---|---|
| Accelerator | **GPU** (T4 x2 hoặc P100; notebook dùng 1 GPU) |
| Internet | **On** (tải `roberta-base` ~500 MB và WikiHow `wikihowAll.csv` ~620 MB vào bộ nhớ tạm, không vào Output) |
| Output | `/kaggle/working/pp34_v2/`: dự đoán dev (+ test kín), lịch sử loss/F1, biểu đồ, E(X,Y), checkpoint tốt nhất của 6 hệ thống (fp16) |
| Thời gian | **ước tính** 3–5 giờ GPU (chưa đo; 25 lượt huấn luyện ≤ 8 epoch + PP3 + CV). Dừng sớm (patience 3); `MAX_HOURS`=9 để kết thúc êm trước giới hạn phiên Kaggle |
| Checkpoint | Mỗi epoch lưu `state/inprogress_*.pt` (mô hình + optimizer + scheduler + RNG). Bị ngắt thì **Run All lại trong cùng phiên** sẽ tiếp tục đúng epoch đó. **Sang phiên mới:** Add Input → *Notebooks* → output phiên bản trước của chính notebook này → Run All (phần đã xong được bỏ qua). Checkpoint TỐT NHẤT của mọi hệ thống (seed đầu): `checkpoints/*_best.pt` |

**Chạy:** Save Version → **Save & Run All (Commit)**. **Gửi lại:** khối `=== GỬI LẠI ===` + `pp34_v2/summary_05.json`.

## 1. Cấu hình

In [ ]:
import os, sys, json, time, platform, shutil, hashlib, csv, zipfile, random, statistics, urllib.request, warnings
from collections import Counter, defaultdict
from pathlib import Path
warnings.filterwarnings("ignore", category=SyntaxWarning)   # stemmer.py gốc có chuỗi '\m' cũ

SEED = 2026
RUN_TAG = "pp34_v2"
INPUT_ROOT = Path(os.environ.get("PROPARA_INPUT_ROOT", "/kaggle/input"))
WORK_ROOT  = Path(os.environ.get("PROPARA_WORK_ROOT", "/kaggle/working"))
OUT = WORK_ROOT / RUN_TAG
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()

ENCODER_ID   = os.environ.get("PROPARA_ENCODER_ID", "FacebookAI/roberta-base")
ENCODER_REV  = os.environ.get("PROPARA_ENCODER_REV", "e2da8e2f811d1448a5b465c236feacd80ffbac7b")
WIKIHOW_URL  = "https://huggingface.co/datasets/Lajvi/wikihowAll/resolve/5e44fed0e204c27ad985e6292e55e311504d4fe5/wikihowAll.csv"
WIKIHOW_SHA  = "8ebf5bd485275af895aa83ea3e55c416fe76aa4fefc58e0b9e6d344852bde5aa"
WIKIHOW_LOCAL = os.environ.get("PROPARA_WIKIHOW_LOCAL", "")        # chỉ dùng khi kiểm thử cục bộ
VARIANTS     = ["M0", "CRF_NOPAIR", "CRF_NOE", "CRF_RANDE", "CRF_FULL"]
SEEDS        = [int(x) for x in os.environ.get("PROPARA_SEEDS", "13,42,2026").split(",")]
EPOCHS       = int(os.environ.get("PROPARA_EPOCHS", "8"))
RUN_CV       = os.environ.get("PROPARA_RUN_CV", "1") == "1"
CV_VARIANTS  = ["M0", "CRF_FULL"]                                    # M0 cho cả M0 và M0c
LIMIT_TRAIN  = int(os.environ.get("PROPARA_LIMIT_TRAIN", "0"))       # >0 chỉ để kiểm thử cục bộ
MAX_HOURS    = float(os.environ.get("PROPARA_MAX_HOURS", "9"))   # không bắt đầu lượt huấn luyện mới sau mốc này → kết thúc êm, kịp lưu Output
PATIENCE     = int(os.environ.get("PROPARA_PATIENCE", "3"))      # dừng sớm: PAIR_F1 dev không tăng 3 epoch liên tiếp
STOP_AFTER   = int(os.environ["PROPARA_STOP_AFTER_EPOCHS"]) if os.environ.get("PROPARA_STOP_AFTER_EPOCHS") else None  # chỉ để kiểm thử
N_BOOT       = 10000

PIN_PROPARA = "51a3cccbedcbe5f92276f9925e0ce1e6ec4247bb"   # allenai/propara (2019-08-30)
PIN_LEADER  = "60d5be31b5a7d36f29d13223ece29a8d2bfa8b5f"   # allenai/aristo-leaderboard (2021-05-17)
_GH = "https://raw.githubusercontent.com/allenai"
RAW_FILES = {
    "grids.v1.train.json": (f"{_GH}/propara/{PIN_PROPARA}/data/emnlp18/grids.v1.train.json",
        "6dc048d09c532ef906ba37d5ab814f54c54e56f230184029912eabe415de225d", ["grids.v1.train.json"]),
    "grids.v1.dev.json": (f"{_GH}/propara/{PIN_PROPARA}/data/emnlp18/grids.v1.dev.json",
        "580ddc66d86a89c747d3e7f2da5cca0eee95444cca66b60b345fe29c93c3fef6", ["grids.v1.dev.json"]),
    "grids.v1.test.json": (f"{_GH}/propara/{PIN_PROPARA}/data/emnlp18/grids.v1.test.json",
        "cf239a45d3eb203aba09ec392cffc81812e5c6fde69d782b01d068efd9e6f370", ["grids.v1.test.json"]),
    "naacl18.partition.paraid.tsv": (f"{_GH}/propara/{PIN_PROPARA}/data/emnlp18/naacl18.partition.paraid.tsv",
        "9fecf07296c58b195ae778943fc0b4d38ea489ec71e19045bd8730460175de44", ["naacl18.partition.paraid.tsv"]),
    "gold-full-grids.v3.tsv": (f"{_GH}/propara/{PIN_PROPARA}/data/naacl18/gold-full-grids.v3.tsv",
        "8eff6e8fb066869d9d0c0fe18e3d994eb3078caa0a3a1be98f716e8a79dbe0aa", ["gold-full-grids.v3.tsv"]),
    "leaderboard_answers_train.tsv": (f"{_GH}/aristo-leaderboard/{PIN_LEADER}/propara/data/train/answers.tsv",
        "2f2b014aa071583f9180cd8487c855438821abe1c9aae5474ac685854934e8cd", ["leaderboard_answers_train.tsv", "train/answers.tsv"]),
    "leaderboard_answers_dev.tsv": (f"{_GH}/aristo-leaderboard/{PIN_LEADER}/propara/data/dev/answers.tsv",
        "a496afe5bb2b81ad79b0da7a935f81d5d0a6b4cefc89d865ba09fece38a30217", ["leaderboard_answers_dev.tsv", "dev/answers.tsv"]),
    "leaderboard_answers_test.tsv": (f"{_GH}/aristo-leaderboard/{PIN_LEADER}/propara/data/test/answers.tsv",
        "cf1a71f38ccad7609e54b1dcf5cbc6e342ea3d2513490c0c09883d5bd80625d6", ["leaderboard_answers_test.tsv", "test/answers.tsv"]),
}
SPLITS = ("train", "dev", "test")

## 2. Môi trường + GPU

In [ ]:
def _internet_ok(url="https://raw.githubusercontent.com", timeout=6):
    try:
        urllib.request.urlopen(url, timeout=timeout)
        return True
    except Exception as e:
        return f"không truy cập được: {type(e).__name__}"

ENV = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "on_kaggle": Path("/kaggle").exists() and "PROPARA_INPUT_ROOT" not in os.environ,
    "cpu_count": os.cpu_count(),
    "disk_free_gb": round(shutil.disk_usage(WORK_ROOT).free / 1e9, 1),
    "gpu_present": bool(shutil.which("nvidia-smi")),
    "internet": _internet_ok(),
}
try:
    with open("/proc/meminfo") as f:
        ENV["ram_total_gb"] = round(int(f.readline().split()[1]) / 1e6, 1)
except OSError:
    ENV["ram_total_gb"] = None
for k, v in ENV.items():
    print(f"{k:14s}: {v}")

import importlib
import torch, transformers
ENV.update({"torch": torch.__version__, "transformers": transformers.__version__,
            "gpus": [(torch.cuda.get_device_name(i), round(torch.cuda.get_device_properties(i).total_memory / 2**30, 1)) for i in range(torch.cuda.device_count())]})
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("GPU:", ENV["gpus"], "| torch", ENV["torch"], "| transformers", ENV["transformers"], "| device", DEVICE)
if DEVICE == "cpu" and not LIMIT_TRAIN:
    raise RuntimeError("Không thấy GPU. Settings → Accelerator → GPU.")

## 3. Nạp mã nhúng + kiểm thử (core, bộ chấm, CRF so với liệt kê vét cạn)

In [ ]:
MODULES = {}
MODULES['propara_core.py'] = r'''
"""propara_core — chuẩn hoá ProPara, suy nhãn, cờ nhắc, cặp vàng, điểm quyết định.

Phiên bản: core_v1 (notebook 00_propara_data_audit). Chỉ dùng thư viện chuẩn của Python.

Quy ước chỉ số:
  * states[i] có độ dài T+1: states[i][0] là trạng thái ban đầu, states[i][t] là trạng thái SAU câu t.
  * labels[i], mentions[i], implicit[i] có độ dài T; phần tử [t-1] ứng với câu t (đánh số từ 1).
  * sentence_id luôn đánh số từ 1 như bộ chấm chính thức của ProPara.
"""
from __future__ import annotations

import hashlib
import json
import re
from collections import Counter, defaultdict
from typing import Dict, Iterable, List, Optional, Tuple

CORE_VERSION = "core_v1"

NONE, CREATE, DESTROY, MOVE = "NONE", "CREATE", "DESTROY", "MOVE"
LABELS = (CREATE, DESTROY, MOVE, NONE)
NO_LOC = "-"   # không tồn tại
UNK_LOC = "?"  # tồn tại nhưng không rõ vị trí

# Từ rỗng bỏ qua khi xét cờ nhắc. Danh sách này được KHOÁ cùng core_v1;
# với danh sách này + MENTION_MODE="report" ta tái hiện đúng Bảng 1 của báo cáo.
STOPWORDS = frozenset(
    "a an the of in on at to for from by with and or its it this that these those "
    "is are be into onto some any other another their his her your our".split()
)

_TOKEN_RE = re.compile(r"[a-z0-9]+")


# ----------------------------------------------------------------------------- tiện ích
def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


def tokenize(text: str) -> List[str]:
    return _TOKEN_RE.findall(text.lower())


def split_aliases(participant: str) -> List[str]:
    """'water vapor ; vapor' -> ['water vapor', 'vapor'] (giống evaluator: tách theo ';')."""
    return [p.strip() for p in participant.split(";") if p.strip()]


# ----------------------------------------------------------------------------- nhãn
def derive_label(before: str, after: str) -> str:
    """Công thức (1) của báo cáo."""
    if before == after:
        return NONE
    if before == NO_LOC and after != NO_LOC:
        return CREATE
    if before != NO_LOC and after == NO_LOC:
        return DESTROY
    return MOVE


def exists(state: str) -> bool:
    """'?' = tồn tại nhưng không rõ vị trí; chỉ '-' là không tồn tại."""
    return state != NO_LOC


VALID_LABELS = {False: {CREATE, NONE}, True: {DESTROY, MOVE, NONE}}  # công thức (15)


def count_constraint_violations(labels_by_entity: List[List[str]],
                                init_exists: List[bool]) -> Tuple[int, int]:
    """Số ô vi phạm ràng buộc tồn tại khi trạng thái TỰ TIẾN HOÁ theo nhãn dự đoán (công thức 28).

    Trả về (số ô vi phạm, số ô). Khi vi phạm, trạng thái vẫn cập nhật theo nhãn
    (CREATE -> tồn tại, DESTROY -> không tồn tại) để không che lỗi tiếp theo.
    """
    bad = total = 0
    for labs, e in zip(labels_by_entity, init_exists):
        for y in labs:
            total += 1
            if y not in VALID_LABELS[e]:
                bad += 1
            if y == CREATE:
                e = True
            elif y == DESTROY:
                e = False
    return bad, total


# ----------------------------------------------------------------------------- cờ nhắc
def _variants(word: str, mode: str) -> set:
    v = {word, word + "s"}
    if word.endswith("s"):
        v.add(word[:-1])
    if mode == "extended":  # nhạy hơn với số nhiều -es/-ies (dùng cho phân tích độ nhạy)
        v.add(word + "es")
        if word.endswith("es"):
            v.add(word[:-2])
        if word.endswith("ies"):
            v.add(word[:-3] + "y")
        if word.endswith("y"):
            v.add(word[:-1] + "ies")
    return v


def is_mentioned(participant: str, sentence: str, mode: str = "report") -> bool:
    """m_it = 1 nếu có ít nhất một từ nội dung của tên thực thể (mọi tên gọi khác,
    dạng số ít/số nhiều) xuất hiện trong câu.

    mode="report": khớp w, w+s, w bỏ 's' — tái hiện đúng Bảng 1 của báo cáo.
    mode="extended": thêm -es/-ies (echo/echoes, virus/viruses) — chỉ để đo độ nhạy.
    """
    if mode not in ("report", "extended"):
        raise ValueError(f"mode không hợp lệ: {mode}")
    sent_tokens = set(tokenize(sentence))
    for alias in split_aliases(participant):
        for w in tokenize(alias):
            if w in STOPWORDS:
                continue
            if _variants(w, mode) & sent_tokens:
                return True
    return False


# ----------------------------------------------------------------------------- bản ghi chuẩn
def build_record(raw: dict, split: str, topic: Optional[str] = None,
                 mention_mode: str = "report") -> dict:
    """Chuyển một dòng grids.v1.*.json thành bản ghi chuẩn (không sửa nhãn gốc)."""
    sents = list(raw["sentence_texts"])
    parts = list(raw["participants"])
    states = [list(s) for s in raw["states"]]
    T = len(sents)
    labels, mentions, implicit, existence = [], [], [], []
    for i, p in enumerate(parts):
        st = states[i]
        labs = [derive_label(st[t - 1], st[t]) for t in range(1, len(st))]
        men = [is_mentioned(p, sents[t - 1], mention_mode) for t in range(1, T + 1)] if len(st) == T + 1 else []
        labels.append(labs)
        mentions.append(men)
        implicit.append([(y != NONE) and (not m) for y, m in zip(labs, men)])
        existence.append([exists(s) for s in st])
    return {
        "paragraph_id": str(raw["para_id"]),
        "split": split,
        "topic": topic,
        "sentences": sents,
        "entities": parts,
        "entity_aliases": [split_aliases(p) for p in parts],
        "states": states,
        "existence": existence,
        "labels": labels,
        "mentions": mentions,
        "implicit": implicit,
        "mention_mode": mention_mode,
        "core_version": CORE_VERSION,
    }


def validate_record(rec: dict) -> List[str]:
    """Kiểm tra cấu trúc; trả về danh sách lỗi (rỗng = hợp lệ)."""
    issues = []
    pid = rec["paragraph_id"]
    T, n = len(rec["sentences"]), len(rec["entities"])
    if T == 0:
        issues.append(f"{pid}: không có câu")
    if n == 0:
        issues.append(f"{pid}: không có thực thể")
    if len(rec["states"]) != n:
        issues.append(f"{pid}: số hàng states ({len(rec['states'])}) != số thực thể ({n})")
    for t, s in enumerate(rec["sentences"], 1):
        if not s or not s.strip():
            issues.append(f"{pid}: câu {t} rỗng")
    lowered = [p.strip().lower() for p in rec["entities"]]
    for p, c in Counter(lowered).items():
        if c > 1:
            issues.append(f"{pid}: thực thể trùng tên '{p}'")
    for i, st in enumerate(rec["states"]):
        if len(st) != T + 1:
            issues.append(f"{pid}: thực thể {i} có {len(st)} trạng thái, cần {T + 1}")
        for t, v in enumerate(st):
            if not isinstance(v, str) or v.strip() == "":
                issues.append(f"{pid}: trạng thái rỗng tại thực thể {i}, bước {t}")
            elif v != v.strip():
                issues.append(f"{pid}: trạng thái có khoảng trắng thừa '{v}' (thực thể {i}, bước {t})")
    return issues


# ----------------------------------------------------------------------------- cặp vàng
def destroyed_created_at(rec: dict, t: int) -> Tuple[List[int], List[int]]:
    D = [i for i in range(len(rec["entities"])) if rec["labels"][i][t - 1] == DESTROY]
    C = [i for i in range(len(rec["entities"])) if rec["labels"][i][t - 1] == CREATE]
    return D, C


def gold_pairs_report(rec: dict) -> List[dict]:
    """Công thức (3): GoldPairs(t) = D_t x C_t. Đây là nhãn SUY TỪ TRẠNG THÁI,
    không phải quan hệ nhân quả đã được người xác nhận (pair_gold_type)."""
    out = []
    for t in range(1, len(rec["sentences"]) + 1):
        D, C = destroyed_created_at(rec, t)
        for x in D:
            for y in C:
                out.append({
                    "paragraph_id": rec["paragraph_id"],
                    "split": rec["split"],
                    "sentence_id": t,
                    "source": rec["entities"][x],
                    "target": rec["entities"][y],
                    "source_idx": x,
                    "target_idx": y,
                    "source_location_before": rec["states"][x][t - 1],
                    "target_location_after": rec["states"][y][t],
                    "source_implicit": bool(rec["implicit"][x][t - 1]),
                    "target_implicit": bool(rec["implicit"][y][t - 1]),
                    "n_destroy_in_sentence": len(D),
                    "n_create_in_sentence": len(C),
                    "pair_gold_type": "derived_from_states",
                })
    return out


def official_q3_conversions(rec: dict) -> List[dict]:
    """Bản sao logic Process.conversions() của evaluator chính thức
    (allenai/aristo-leaderboard/propara, commit 60d5be3), GIỮ NGUYÊN cả điều kiện
    `step_id < num_steps - 1`. Mỗi câu cho tối đa MỘT conversion, gộp mọi thực thể
    bị huỷ/tạo bằng ' AND '; có luật ghép với câu kế tiếp (rule b).
    Notebook 01 sẽ đối chiếu hàm này với mã evaluator gốc."""
    T = len(rec["sentences"])
    names = [" OR ".join(split_aliases(p)) for p in rec["entities"]]

    def loc(v):  # evaluator đổi '?' -> 'unk', '-' -> 'null'
        return {"?": "unk", "-": "null"}.get(v, v)

    def created(t):
        idx = [i for i in range(len(names)) if rec["states"][i][t - 1] == NO_LOC and rec["states"][i][t] != NO_LOC]
        return [names[i] for i in idx], [loc(rec["states"][i][t]) for i in idx]

    def destroyed(t):
        idx = [i for i in range(len(names)) if rec["states"][i][t - 1] != NO_LOC and rec["states"][i][t] == NO_LOC]
        return [names[i] for i in idx], [loc(rec["states"][i][t - 1]) for i in idx]

    conv = []
    for t in range(1, T + 1):
        c, cl = created(t)
        d, dl = destroyed(t)
        if c and d:
            conv.append({"step_id": t, "destroyed": " AND ".join(d), "created": " AND ".join(c),
                         "locations": sorted(set(cl + dl)), "rule": "same_step"})
        elif d and t < T - 1:
            c2, cl2 = created(t + 1)
            d2, _ = destroyed(t + 1)
            cnd = [x for x in c2 if x not in set(d)]
            if not d2 and cnd:
                conv.append({"step_id": t, "destroyed": " AND ".join(d), "created": " AND ".join(sorted(set(cnd))),
                             "locations": sorted(set(cl2 + dl)), "rule": "next_step_create"})
        elif c and t < T - 1:
            c2, _ = created(t + 1)
            d2, dl2 = destroyed(t + 1)
            dnc = [x for x in d2 if x not in set(c)]
            if not c2 and dnc:
                conv.append({"step_id": t, "destroyed": " AND ".join(sorted(set(dnc))), "created": " AND ".join(c),
                             "locations": sorted(set(cl + dl2)), "rule": "next_step_destroy"})
    return conv


def to_action_rows(rec: dict) -> List[Tuple[str, int, str, str, str, str]]:
    """Dạng 'action file' của evaluator: (process_id, step, participant, action, before, after)."""
    rows = []
    for t in range(1, len(rec["sentences"]) + 1):
        for i, p in enumerate(rec["entities"]):
            rows.append((rec["paragraph_id"], t, p, rec["labels"][i][t - 1],
                         rec["states"][i][t - 1], rec["states"][i][t]))
    return rows


# ----------------------------------------------------------------------------- điểm quyết định ngầm
def decision_points(rec: dict) -> List[dict]:
    """Mục 2.2: bộ (đoạn, câu, nhãn) có >=1 ô vàng ngầm và >=2 ứng viên.
    Ứng viên DESTROY = thực thể đang tồn tại trước câu; CREATE = chưa tồn tại;
    thực thể vàng ĐƯỢC NHẮC bị loại khỏi tập ứng viên."""
    out = []
    n = len(rec["entities"])
    for t in range(1, len(rec["sentences"]) + 1):
        for lab in (DESTROY, CREATE):
            need_exist = lab == DESTROY
            gold = [i for i in range(n) if rec["labels"][i][t - 1] == lab]
            gold_implicit = [i for i in gold if rec["implicit"][i][t - 1]]
            gold_mentioned = [i for i in gold if not rec["implicit"][i][t - 1]]
            if not gold_implicit:
                continue
            cands = [i for i in range(n)
                     if rec["existence"][i][t - 1] == need_exist and i not in gold_mentioned]
            if len(cands) < 2:
                continue
            out.append({
                "paragraph_id": rec["paragraph_id"], "split": rec["split"], "topic": rec.get("topic"),
                "sentence_id": t, "label": lab,
                "candidates": cands, "gold": gold_implicit,
                "n_candidates": len(cands), "n_gold": len(gold_implicit),
                "random_acc": len(gold_implicit) / len(cands),
            })
    return out


# ----------------------------------------------------------------------------- PP2 / PP1 (đếm)
def pp2_counts(rec: dict) -> Dict[str, int]:
    """Công thức (7) và (12): mọi cặp có thứ tự (X != Y) mỗi câu, và số cặp còn lại
    sau lọc bằng TỒN TẠI VÀNG (X tồn tại trước câu, Y chưa tồn tại). Đây là MỨC TRẦN."""
    T, n = len(rec["sentences"]), len(rec["entities"])
    total = T * n * (n - 1)
    kept = 0
    gold = {(g["source_idx"], g["target_idx"], g["sentence_id"]) for g in gold_pairs_report(rec)}
    gold_kept = 0
    for t in range(1, T + 1):
        ex = [rec["existence"][i][t - 1] for i in range(n)]
        for x in range(n):
            for y in range(n):
                if x != y and ex[x] and not ex[y]:
                    kept += 1
                    if (x, y, t) in gold:
                        gold_kept += 1
    return {"all_pairs": total, "kept_gold_existence": kept,
            "gold_pairs": len(gold), "gold_kept": gold_kept, "label_questions": 4 * T * n}


def cooccurrence_filter_coverage(rec: dict) -> Dict[str, int]:
    """Thử tái hiện câu 'lọc theo đồng xuất hiện trong câu chỉ giữ 74,7% cặp vàng và 18,9%
    cặp có nguồn ngầm' (mục 4.2). Báo cáo KHÔNG định nghĩa rõ, nên đo hai cách hiểu:
      same_sentence: X và Y cùng được nhắc trong chính câu t;
      any_sentence:  X và Y cùng được nhắc trong ít nhất một câu nào đó của đoạn."""
    out = Counter()
    T = len(rec["sentences"])
    for g in gold_pairs_report(rec):
        x, y, t = g["source_idx"], g["target_idx"], g["sentence_id"]
        same = rec["mentions"][x][t - 1] and rec["mentions"][y][t - 1]
        anys = any(rec["mentions"][x][k] and rec["mentions"][y][k] for k in range(T))
        out["gold"] += 1
        out["same_sentence"] += int(same)
        out["any_sentence"] += int(anys)
        if g["source_implicit"]:
            out["gold_src_implicit"] += 1
            out["same_sentence_src_implicit"] += int(same)
            out["any_sentence_src_implicit"] += int(anys)
    return dict(out)


# ----------------------------------------------------------------------------- lift (Bảng 2)
def lift_table(records: Iterable[dict]) -> Dict[Tuple[str, str], Dict[str, float]]:
    """Công thức (2). Ô hợp lệ: CREATE -> thực thể chưa tồn tại trước câu;
    DESTROY, MOVE -> đang tồn tại. Chỉ xét ô KHÔNG được nhắc (m_it = 0).
    B = 'trong câu t có thực thể j != i được nhắc và nhận nhãn B'."""
    cnt = defaultdict(Counter)
    for rec in records:
        n = len(rec["entities"])
        for t in range(1, len(rec["sentences"]) + 1):
            for i in range(n):
                if rec["mentions"][i][t - 1]:
                    continue
                y = rec["labels"][i][t - 1]
                e = rec["existence"][i][t - 1]
                for A in (CREATE, DESTROY, MOVE):
                    if (A == CREATE) == e:  # ô không hợp lệ cho nhãn A
                        continue
                    for B in (CREATE, DESTROY, MOVE):
                        sig = any(j != i and rec["mentions"][j][t - 1] and rec["labels"][j][t - 1] == B
                                  for j in range(n))
                        cnt[(A, B)][("sig" if sig else "nosig", y == A)] += 1
    res = {}
    for (A, B), c in cnt.items():
        a1, n1 = c[("sig", True)], c[("sig", True)] + c[("sig", False)]
        a0, n0 = c[("nosig", True)], c[("nosig", True)] + c[("nosig", False)]
        p1 = a1 / n1 if n1 else float("nan")
        p0 = a0 / n0 if n0 else float("nan")
        res[(A, B)] = {"lift": (p1 / p0) if p0 and p0 == p0 else float("nan"),
                       "n_implicit_A_with_B": a1, "p_sig": p1, "p_nosig": p0}
    return res


# ----------------------------------------------------------------------------- nạp dữ liệu
def load_jsonl(path: str) -> List[dict]:
    with open(path, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def load_topics_from_naacl_grid(path: str) -> Dict[str, str]:
    """gold-full-grids.v3.tsv: dòng có cột 3 bắt đầu 'PROMPT:' cho biết chủ đề quy trình."""
    topics = {}
    with open(path, encoding="utf-8") as f:
        for line in f:
            c = line.rstrip("\n").split("\t")
            if len(c) > 2 and c[2].startswith("PROMPT:"):
                topics[c[0].strip()] = c[2][len("PROMPT:"):].strip()
    return topics


def load_action_file(path: str) -> List[Tuple[str, int, str, str, str, str]]:
    """answers.tsv của leaderboard: (pid, step, participant, action, before, after).
    Trả về DANH SÁCH (không dùng dict) để không âm thầm gộp các thực thể trùng tên."""
    out = []
    with open(path, encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            if not line.strip():
                continue
            c = line.rstrip("\n\r").split("\t")
            if len(c) < 6:
                raise ValueError(f"{path}:{ln}: cần 6 cột, có {len(c)}")
            out.append((c[0], int(c[1]), c[2], c[3], c[4], c[5]))
    return out


def compare_action_rows(official: List[tuple], derived: List[tuple]) -> Dict[str, object]:
    """So sánh dạng đa tập theo (pid, step): bỏ qua thứ tự dòng, không gộp trùng tên."""
    a, b = defaultdict(list), defaultdict(list)
    for r in official:
        a[(str(r[0]), int(r[1]))].append(tuple(r[2:6]))
    for r in derived:
        b[(str(r[0]), int(r[1]))].append(tuple(r[2:6]))
    keys = set(a) | set(b)
    bad = sorted(k for k in keys if sorted(a[k]) != sorted(b[k]))
    return {"n_official_rows": len(official), "n_derived_rows": len(derived),
            "n_mismatched_steps": len(bad), "examples": bad[:5]}


# ----------------------------------------------------------------------------- chia chéo & mẫu kiểm tra
def stable_hash(s: str) -> int:
    return int(hashlib.md5(s.encode("utf-8")).hexdigest()[:8], 16)


def cv_folds_by_topic(records: List[dict], k: int = 5, seed: int = 2026) -> Dict[str, int]:
    """Chia k phần theo CHỦ ĐỀ (prompt), tất định. Cân bằng tham lam đồng thời
    số điểm quyết định ngầm, số cặp vàng và số đoạn (mỗi đại lượng chuẩn hoá theo tổng).
    Trả về {paragraph_id: fold}."""
    by_topic = defaultdict(list)
    for r in records:
        by_topic[r.get("topic") or f"__no_topic__{r['paragraph_id']}"].append(r)
    stats = []
    for tp, rs in by_topic.items():
        v = (sum(len(decision_points(r)) for r in rs), sum(len(gold_pairs_report(r)) for r in rs), len(rs))
        stats.append((v, stable_hash(f"{seed}:{tp}"), tp))
    tot = [max(sum(s[0][d] for s in stats), 1) for d in range(3)]
    stats.sort(key=lambda x: (-sum(x[0][d] / tot[d] for d in range(3)), x[1]))
    load = [[0, 0, 0] for _ in range(k)]
    topic_fold = {}
    for v, _, tp in stats:
        f = min(range(k), key=lambda j: (max((load[j][d] + v[d]) / tot[d] for d in range(3)),
                                         sum(load[j][d] / tot[d] for d in range(3)), j))
        topic_fold[tp] = f
        for d in range(3):
            load[f][d] += v[d]
    return {r["paragraph_id"]: topic_fold[r.get("topic") or f"__no_topic__{r['paragraph_id']}"] for r in records}


def gold_audit_sample(pairs: List[dict], quotas: Dict[str, int], seed: int = 2026) -> List[dict]:
    """Lấy mẫu có chủ đích, tất định, cho người kiểm tra quan hệ X->Y.
    Nhóm (ưu tiên theo thứ tự): source_implicit, target_implicit, multi_pair, regular."""
    def group(p):
        if p["source_implicit"]:
            return "source_implicit"
        if p["target_implicit"]:
            return "target_implicit"
        if p["n_destroy_in_sentence"] * p["n_create_in_sentence"] > 1:
            return "multi_pair"
        return "regular"
    out = []
    for g, q in quotas.items():
        pool = [p for p in pairs if group(p) == g]
        pool.sort(key=lambda p: stable_hash(f"{seed}:{p['paragraph_id']}:{p['sentence_id']}:{p['source_idx']}:{p['target_idx']}"))
        for p in pool[:q]:
            out.append(dict(p, audit_group=g))
    return out
'''
MODULES['propara_core_tests.py'] = r'''
"""Kiểm thử đơn vị cho propara_core (chạy được bằng pytest hoặc run_all()).

Dữ liệu ở đây là ví dụ tự dựng (không phải mẫu ProPara thật).
"""
import propara_core as pc


def _toy(states, sents=None, parts=None, pid="toy"):
    n, T = len(states), len(states[0]) - 1
    raw = {"para_id": pid,
           "sentence_texts": sents or [f"Sentence {t}." for t in range(1, T + 1)],
           "participants": parts or [f"entity{i}" for i in range(n)],
           "states": states}
    return pc.build_record(raw, "train", topic="toy topic")


def test_derive_label_all_cases():
    assert pc.derive_label("leaf", "leaf") == pc.NONE
    assert pc.derive_label("-", "-") == pc.NONE
    assert pc.derive_label("?", "?") == pc.NONE
    assert pc.derive_label("-", "leaf") == pc.CREATE
    assert pc.derive_label("-", "?") == pc.CREATE          # tạo ra nhưng không rõ vị trí
    assert pc.derive_label("leaf", "-") == pc.DESTROY
    assert pc.derive_label("?", "-") == pc.DESTROY          # '?' vẫn là tồn tại
    assert pc.derive_label("leaf", "root") == pc.MOVE
    assert pc.derive_label("?", "leaf") == pc.MOVE          # quy ước báo cáo: còn lại là MOVE
    assert pc.derive_label("leaf", "?") == pc.MOVE


def test_exists_question_mark_is_existing():
    assert pc.exists("?") and pc.exists("leaf") and not pc.exists("-")


def test_mention_report_mode_singular_plural():
    assert pc.is_mentioned("plants", "The plant dies.")
    assert pc.is_mentioned("plant", "Plants die.")
    assert pc.is_mentioned("carbon dioxide; CO2", "The leaf absorbs co2.")   # tên gọi khác
    assert not pc.is_mentioned("the water", "The sun shines.")                # 'the' là từ rỗng
    assert not pc.is_mentioned("water", "Waterfalls are big.")                # khớp nguyên từ
    # chế độ report KHÔNG bắt -es / -ies; chế độ extended thì có
    assert not pc.is_mentioned("echo", "Echoes are produced.")
    assert pc.is_mentioned("echo", "Echoes are produced.", mode="extended")
    assert pc.is_mentioned("berry", "Berries grow.", mode="extended")


def test_build_record_labels_and_implicit():
    rec = _toy([["leaf", "leaf", "-"], ["-", "-", "leaf"]],
               sents=["The entity0 sits.", "Sugar is made."], parts=["entity0", "sugar"])
    assert rec["labels"] == [[pc.NONE, pc.DESTROY], [pc.NONE, pc.CREATE]]
    assert rec["implicit"][0] == [False, True]    # entity0 bị huỷ ở câu 2 mà không được nhắc
    assert rec["implicit"][1] == [False, False]   # sugar được nhắc
    assert pc.validate_record(rec) == []


def test_gold_pairs_cross_product_and_direction():
    rec = _toy([["a", "-"], ["b", "-"], ["-", "c"], ["-", "d"], ["e", "e"]])
    pairs = pc.gold_pairs_report(rec)
    got = {(p["source_idx"], p["target_idx"], p["sentence_id"]) for p in pairs}
    assert got == {(0, 2, 1), (0, 3, 1), (1, 2, 1), (1, 3, 1)}       # 2 x 2 = 4 cặp
    assert (2, 0, 1) not in got                                       # đúng chiều nguồn -> đích
    assert all(p["pair_gold_type"] == "derived_from_states" for p in pairs)


def test_no_pairs_when_only_destroy():
    rec = _toy([["a", "-"], ["b", "b"]])
    assert pc.gold_pairs_report(rec) == []


def test_official_conversion_same_step_and_next_step_rule():
    # câu 1: huỷ e0, không tạo; câu 2: chỉ tạo e1 -> rule b gán conversion về câu 1
    rec = _toy([["x", "-", "-", "-", "-"], ["-", "-", "y", "y", "y"]])
    conv = pc.official_q3_conversions(rec)
    assert len(conv) == 1 and conv[0]["step_id"] == 1 and conv[0]["rule"] == "next_step_create"
    # cùng câu: một conversion gộp mọi D và C
    rec2 = _toy([["a", "-"], ["b", "-"], ["-", "c"]])
    conv2 = pc.official_q3_conversions(rec2)
    assert len(conv2) == 1 and conv2[0]["destroyed"] == "entity0 AND entity1"
    assert len(pc.gold_pairs_report(rec2)) == 2     # trong khi định nghĩa cặp của báo cáo cho 2 cặp


def test_official_conversion_keeps_evaluator_off_by_one():
    # T=3: huỷ ở câu 2, tạo ở câu 3. Evaluator chỉ xét rule b khi step < T-1 = 2 -> KHÔNG có conversion
    rec = _toy([["x", "x", "-", "-"], ["-", "-", "-", "y"]])
    assert pc.official_q3_conversions(rec) == []


def test_decision_points_definition():
    # câu 1: e0 bị huỷ ngầm; ứng viên DESTROY = các thực thể đang tồn tại (e0, e1) -> 1 điểm quyết định
    rec = _toy([["a", "-"], ["b", "b"], ["-", "-"]],
               sents=["Something happens."], parts=["alpha", "beta", "gamma"])
    dps = pc.decision_points(rec)
    assert len(dps) == 1 and dps[0]["label"] == pc.DESTROY
    assert dps[0]["candidates"] == [0, 1] and dps[0]["gold"] == [0]
    assert abs(dps[0]["random_acc"] - 0.5) < 1e-12


def test_decision_point_excludes_mentioned_gold_and_needs_two_candidates():
    # beta bị huỷ và ĐƯỢC NHẮC -> bị loại khỏi ứng viên; còn alpha (ngầm) + gamma
    rec = _toy([["a", "-"], ["b", "-"], ["c", "c"]],
               sents=["The beta breaks."], parts=["alpha", "beta", "gamma"])
    dps = pc.decision_points(rec)
    assert len(dps) == 1 and dps[0]["candidates"] == [0, 2]
    # chỉ còn 1 ứng viên -> không phải điểm quyết định
    rec2 = _toy([["a", "-"], ["b", "-"]], sents=["The beta breaks."], parts=["alpha", "beta"])
    assert pc.decision_points(rec2) == []


def test_constraint_violations_self_evolving():
    # DESTROY khi chưa tồn tại, rồi CREATE hợp lệ
    bad, tot = pc.count_constraint_violations([[pc.DESTROY, pc.CREATE]], [False])
    assert (bad, tot) == (1, 2)
    # DESTROY hai lần liên tiếp mà không CREATE lại -> lần 2 vi phạm
    bad, _ = pc.count_constraint_violations([[pc.DESTROY, pc.DESTROY]], [True])
    assert bad == 1
    # chuỗi hợp lệ
    bad, _ = pc.count_constraint_violations([[pc.NONE, pc.CREATE, pc.MOVE, pc.DESTROY]], [False])
    assert bad == 0


def test_gold_sequences_never_violate_constraints():
    rec = _toy([["a", "-", "-", "b", "?"], ["-", "x", "y", "-", "-"]])
    init = [rec["existence"][i][0] for i in range(2)]
    assert pc.count_constraint_violations(rec["labels"], init)[0] == 0


def test_validate_record_catches_errors():
    raw = {"para_id": "bad", "sentence_texts": ["One.", ""], "participants": ["a", "A"],
           "states": [["x", "x", "x"], ["x", ""]]}
    issues = pc.validate_record(pc.build_record(raw, "dev"))
    text = " | ".join(issues)
    assert "câu 2 rỗng" in text and "trùng tên" in text and "cần 3" in text


def test_pp2_counts_gold_ceiling():
    rec = _toy([["a", "-"], ["-", "c"], ["e", "e"]])
    c = pc.pp2_counts(rec)
    assert c["all_pairs"] == 1 * 3 * 2
    assert c["kept_gold_existence"] == 2       # (e0->e1), (e2->e1)
    assert c["gold_kept"] == c["gold_pairs"] == 1


def test_compare_action_rows_ignores_order_but_not_content():
    a = [("1", 1, "x", "NONE", "-", "-"), ("1", 1, "y", "CREATE", "-", "leaf")]
    assert pc.compare_action_rows(a, list(reversed(a)))["n_mismatched_steps"] == 0
    b = [("1", 1, "x", "NONE", "-", "-"), ("1", 1, "y", "CREATE", "-", "root")]
    assert pc.compare_action_rows(a, b)["n_mismatched_steps"] == 1


def test_cv_folds_deterministic_and_grouped():
    recs = [_toy([["a", "-"], ["-", "b"]], pid=str(i)) for i in range(10)]
    for i, r in enumerate(recs):
        r["topic"] = f"topic{i % 4}"
    f1 = pc.cv_folds_by_topic(recs, k=3)
    assert f1 == pc.cv_folds_by_topic(recs, k=3)
    for t in range(4):
        assert len({f1[r["paragraph_id"]] for r in recs if r["topic"] == f"topic{t}"}) == 1


def run_all(verbose=True):
    names = sorted(n for n in globals() if n.startswith("test_"))
    failed = []
    for n in names:
        try:
            globals()[n]()
            if verbose:
                print(f"  PASS  {n}")
        except Exception as e:  # noqa: BLE001 — báo lỗi rõ ràng rồi tổng hợp
            failed.append((n, repr(e)))
            print(f"  FAIL  {n}: {e!r}")
    return len(names), failed
'''
MODULES['propara_metrics.py'] = r'''
"""propara_metrics — bộ chấm chung cho mọi hệ thống (heuristic, Qwen, mô hình nhóm).

Phiên bản: metrics_v1. Phụ thuộc: propara_core (core_v1). Chỉ dùng thư viện chuẩn.

ĐỊNH DẠNG DỰ ĐOÁN (một dict cho mỗi đoạn):
  {
    "paragraph_id": "1002",
    "labels": [[nhãn câu 1..T] cho thực thể 0], ..., [..] cho thực thể n-1]   # BẮT BUỘC, n x T
    "exists_initial": [bool]*n            # tuỳ chọn; thiếu thì suy từ nhãn (xem infer_initial_existence)
    "location_initial": [str]*n           # tuỳ chọn; '-' không tồn tại, '?' không rõ
    "locations_after": [[str]*T]*n        # tuỳ chọn; vị trí SAU câu t
    "pairs": [{"sentence_id", "source_idx", "target_idx"}]   # tuỳ chọn; thiếu thì = D̂_t x Ĉ_t
  }
Thực thể định danh bằng CHỈ SỐ trong danh sách thực thể của đoạn (như dữ liệu chuẩn của notebook 00).
"""
from __future__ import annotations

import io
import math
import random
import re
from collections import Counter, defaultdict
from typing import Callable, Dict, Iterable, List, Optional, Sequence, Tuple

import propara_core as pc

METRICS_VERSION = "metrics_v1"
SLICES = ("all", "src_implicit", "tgt_implicit", "any_implicit", "both_implicit", "both_explicit")


class PredictionError(ValueError):
    """Lỗi cấu trúc của dự đoán (sai kích thước, nhãn lạ, chỉ số thực thể ngoài danh sách...)."""


# ----------------------------------------------------------------------------- kiểm tra cấu trúc
def validate_prediction(pred: dict, rec: dict) -> None:
    n, T = len(rec["entities"]), len(rec["sentences"])
    pid = rec["paragraph_id"]
    if str(pred.get("paragraph_id")) != pid:
        raise PredictionError(f"{pid}: paragraph_id dự đoán = {pred.get('paragraph_id')!r}")
    labs = pred.get("labels")
    if not isinstance(labs, list) or len(labs) != n:
        raise PredictionError(f"{pid}: labels phải có {n} hàng (số thực thể)")
    for i, row in enumerate(labs):
        if not isinstance(row, list) or len(row) != T:
            raise PredictionError(f"{pid}: labels[{i}] phải có {T} phần tử (số câu)")
        for y in row:
            if y not in pc.LABELS:
                raise PredictionError(f"{pid}: nhãn lạ {y!r} ở thực thể {i}")
    for key, length in (("exists_initial", n), ("location_initial", n)):
        if key in pred and pred[key] is not None and len(pred[key]) != length:
            raise PredictionError(f"{pid}: {key} phải có {length} phần tử")
    if pred.get("locations_after") is not None:
        la = pred["locations_after"]
        if len(la) != n or any(len(r) != T for r in la):
            raise PredictionError(f"{pid}: locations_after phải là {n} x {T}")
    for p in pred.get("pairs") or []:
        s, x, y = p.get("sentence_id"), p.get("source_idx"), p.get("target_idx")
        if not (isinstance(s, int) and 1 <= s <= T):
            raise PredictionError(f"{pid}: sentence_id ngoài [1,{T}]: {s!r}")
        for v in (x, y):
            if not (isinstance(v, int) and 0 <= v < n):
                raise PredictionError(f"{pid}: chỉ số thực thể ngoài danh sách: {v!r}")
        if x == y:
            raise PredictionError(f"{pid}: cặp tự thân ({x}->{y})")


def infer_initial_existence(labels_row: Sequence[str]) -> bool:
    """Khi hệ thống không cho tồn tại ban đầu: tồn tại trừ khi thay đổi đầu tiên là CREATE."""
    for y in labels_row:
        if y != pc.NONE:
            return y != pc.CREATE
    return True


def pred_initial_existence(pred: dict) -> List[bool]:
    if pred.get("exists_initial") is not None:
        return [bool(v) for v in pred["exists_initial"]]
    if pred.get("location_initial") is not None:
        return [v != pc.NO_LOC for v in pred["location_initial"]]
    return [infer_initial_existence(r) for r in pred["labels"]]


def gold_as_prediction(rec: dict) -> dict:
    return {"paragraph_id": rec["paragraph_id"], "labels": [list(r) for r in rec["labels"]],
            "exists_initial": [e[0] for e in rec["existence"]],
            "location_initial": [s[0] for s in rec["states"]],
            "locations_after": [s[1:] for s in rec["states"]]}


# ----------------------------------------------------------------------------- cặp
def predicted_pairs(pred: dict, rec: dict) -> set:
    if pred.get("pairs") is not None:
        return {(p["sentence_id"], p["source_idx"], p["target_idx"]) for p in pred["pairs"]}
    out = set()
    n = len(rec["entities"])
    for t in range(1, len(rec["sentences"]) + 1):
        D = [i for i in range(n) if pred["labels"][i][t - 1] == pc.DESTROY]
        C = [i for i in range(n) if pred["labels"][i][t - 1] == pc.CREATE]
        out.update((t, x, y) for x in D for y in C)
    return out


def gold_pairs(rec: dict) -> set:
    return {(g["sentence_id"], g["source_idx"], g["target_idx"]) for g in pc.gold_pairs_report(rec)}


def pair_slice(rec: dict, t: int, x: int, y: int) -> Dict[str, bool]:
    """Lát cắt theo cờ nhắc TÍNH TỪ VĂN BẢN (không cần vàng) nên áp dụng được cho dự đoán."""
    sx = not rec["mentions"][x][t - 1]
    sy = not rec["mentions"][y][t - 1]
    return {"all": True, "src_implicit": sx, "tgt_implicit": sy, "any_implicit": sx or sy,
            "both_implicit": sx and sy, "both_explicit": (not sx) and (not sy)}


def pair_counts(pred: dict, rec: dict) -> Dict[str, Tuple[int, int, int]]:
    """{slice: (tp, n_pred, n_gold)} cho một đoạn."""
    P, G = predicted_pairs(pred, rec), gold_pairs(rec)
    out = {s: [0, 0, 0] for s in SLICES}
    for (t, x, y) in P:
        sl = pair_slice(rec, t, x, y)
        for s in SLICES:
            if sl[s]:
                out[s][1] += 1
                if (t, x, y) in G:
                    out[s][0] += 1
    for (t, x, y) in G:
        sl = pair_slice(rec, t, x, y)
        for s in SLICES:
            if sl[s]:
                out[s][2] += 1
    return {s: tuple(v) for s, v in out.items()}


def prf(tp: int, npred: int, ngold: int) -> Dict[str, float]:
    p = tp / npred if npred else 0.0
    r = tp / ngold if ngold else 0.0
    f = 2 * p * r / (p + r) if p + r else 0.0
    return {"P": p, "R": r, "F1": f, "tp": tp, "n_pred": npred, "n_gold": ngold}


def _norm_loc(s: str) -> str:
    s = s.strip().lower()
    for a in ("the ", "a ", "an ", "its ", "their "):
        if s.startswith(a):
            s = s[len(a):]
    return " ".join(w[:-1] if w.endswith("s") and len(w) > 3 else w for w in re.findall(r"[a-z0-9]+", s))


def pair_loc_counts(pred: dict, rec: dict) -> Tuple[int, int, int]:
    """PAIR_LOC: cặp + vị trí sau của đích (chuẩn hoá nhẹ). Cần locations_after; thiếu thì vị trí = '?'."""
    la = pred.get("locations_after")
    P = {(t, x, y, _norm_loc(la[y][t - 1]) if la else "?") for (t, x, y) in predicted_pairs(pred, rec)}
    G = {(t, x, y, _norm_loc(rec["states"][y][t])) for (t, x, y) in gold_pairs(rec)}
    return len(P & G), len(P), len(G)


# ----------------------------------------------------------------------------- nhãn / tồn tại / ràng buộc
def cell_metrics(pred: dict, rec: dict) -> Dict[str, object]:
    n, T = len(rec["entities"]), len(rec["sentences"])
    conf = Counter()          # (gold, pred)
    imp = Counter()           # (gold_label, correct?) trên ô vàng ngầm
    exist_ok = exist_tot = 0
    init = pred_initial_existence(pred)
    for i in range(n):
        e = init[i]
        for t in range(1, T + 1):
            g, y = rec["labels"][i][t - 1], pred["labels"][i][t - 1]
            conf[(g, y)] += 1
            if rec["implicit"][i][t - 1]:
                imp[(g, g == y)] += 1
            if y == pc.CREATE:
                e = True
            elif y == pc.DESTROY:
                e = False
            exist_ok += int(e == rec["existence"][i][t])
            exist_tot += 1
    viol, tot = pc.count_constraint_violations(pred["labels"], init)
    return {"confusion": conf, "implicit": imp, "exist_ok": exist_ok, "exist_tot": exist_tot,
            "viol": viol, "viol_cells": tot, "n_entities": n}


# ----------------------------------------------------------------------------- tổng hợp
def evaluate(preds: Dict[str, dict], recs: List[dict], strict: bool = True) -> Dict[str, object]:
    """Chấm một hệ thống trên danh sách đoạn. strict=True: thiếu đoạn -> lỗi.
    strict=False: thiếu đoạn -> coi như dự đoán toàn NONE và ghi vào 'missing'."""
    tot = {s: [0, 0, 0] for s in SLICES}
    loc = [0, 0, 0]
    conf, imp = Counter(), Counter()
    ex_ok = ex_tot = viol = viol_cells = n_ent = 0
    missing, per_para = [], {}
    for rec in recs:
        pid = rec["paragraph_id"]
        pred = preds.get(pid)
        if pred is None:
            if strict:
                raise PredictionError(f"Thiếu dự đoán cho đoạn {pid}")
            missing.append(pid)
            pred = all_none_prediction(rec)
        validate_prediction(pred, rec)
        pcnt = pair_counts(pred, rec)
        for s in SLICES:
            for k in range(3):
                tot[s][k] += pcnt[s][k]
        lc = pair_loc_counts(pred, rec)
        for k in range(3):
            loc[k] += lc[k]
        cm = cell_metrics(pred, rec)
        conf.update(cm["confusion"])
        imp.update(cm["implicit"])
        ex_ok += cm["exist_ok"]; ex_tot += cm["exist_tot"]
        viol += cm["viol"]; viol_cells += cm["viol_cells"]; n_ent += cm["n_entities"]
        per_para[pid] = {"pairs": pcnt, "viol": cm["viol"]}
    res = {"metrics_version": METRICS_VERSION, "n_paragraphs": len(recs), "missing": missing}
    for s in SLICES:
        res[f"PAIR@{s}"] = prf(*tot[s])
    res["PAIR_LOC"] = prf(*loc)
    lab = {}
    for L in pc.LABELS:
        tp = conf[(L, L)]
        npred = sum(v for (g, y), v in conf.items() if y == L)
        ngold = sum(v for (g, y), v in conf.items() if g == L)
        lab[L] = prf(tp, npred, ngold)
    res["LABEL"] = lab
    res["LABEL_MACRO_F1_CDM"] = sum(lab[L]["F1"] for L in (pc.CREATE, pc.DESTROY, pc.MOVE)) / 3
    res["IMPLICIT_ACC"] = {L: {"correct": imp[(L, True)], "n": imp[(L, True)] + imp[(L, False)],
                               "acc": (imp[(L, True)] / (imp[(L, True)] + imp[(L, False)]))
                               if imp[(L, True)] + imp[(L, False)] else float("nan")}
                           for L in (pc.CREATE, pc.DESTROY, pc.MOVE)}
    res["EXIST_ACC"] = ex_ok / ex_tot if ex_tot else float("nan")
    res["VIOLATION_RATE_per_entity"] = viol / n_ent if n_ent else float("nan")   # đúng công thức (28)
    res["VIOLATION_RATE_per_cell"] = viol / viol_cells if viol_cells else float("nan")
    res["_per_paragraph"] = per_para
    return res


def all_none_prediction(rec: dict) -> dict:
    n, T = len(rec["entities"]), len(rec["sentences"])
    return {"paragraph_id": rec["paragraph_id"], "labels": [[pc.NONE] * T for _ in range(n)]}


def summary_row(name: str, res: dict, extra: Optional[dict] = None) -> Dict[str, object]:
    r = {"system": name,
         "PAIR_P": res["PAIR@all"]["P"], "PAIR_R": res["PAIR@all"]["R"], "PAIR_F1": res["PAIR@all"]["F1"],
         "PAIR_F1@src_imp": res["PAIR@src_implicit"]["F1"], "PAIR_F1@tgt_imp": res["PAIR@tgt_implicit"]["F1"],
         "PAIR_F1@any_imp": res["PAIR@any_implicit"]["F1"], "PAIR_LOC_F1": res["PAIR_LOC"]["F1"],
         "F1_CREATE": res["LABEL"]["CREATE"]["F1"], "F1_DESTROY": res["LABEL"]["DESTROY"]["F1"],
         "F1_MOVE": res["LABEL"]["MOVE"]["F1"], "IMP_ACC_CREATE": res["IMPLICIT_ACC"]["CREATE"]["acc"],
         "IMP_ACC_DESTROY": res["IMPLICIT_ACC"]["DESTROY"]["acc"], "EXIST_ACC": res["EXIST_ACC"],
         "VIOL_per_entity": res["VIOLATION_RATE_per_entity"]}
    r.update(extra or {})
    return r


# ----------------------------------------------------------------------------- xuất action file cho evaluator chính thức
def to_official_action_rows(pred: dict, rec: dict) -> List[Tuple[str, int, str, str, str, str]]:
    """Chuyển dự đoán thành các dòng hợp lệ theo luật của evaluator (README aristo-leaderboard).
    Vị trí thiếu -> '?'. Dự đoán vi phạm ràng buộc vẫn được ghi ra dòng HỢP LỆ (ví dụ DESTROY khi
    đang không tồn tại -> before='?'), vì định dạng chính thức không biểu diễn được vi phạm.
    VIOLATION_RATE phải đọc từ evaluate(), không từ file này."""
    n, T = len(rec["entities"]), len(rec["sentences"])
    init_e = pred_initial_existence(pred)
    li = pred.get("location_initial")
    la = pred.get("locations_after")
    rows = []
    for i, ent in enumerate(rec["entities"]):
        if not init_e[i]:
            cur = "-"
        else:
            cur = li[i] if li and li[i] not in (None, "", "-") else "?"
        for t in range(1, T + 1):
            y = pred["labels"][i][t - 1]
            want = (la[i][t - 1] if la else None)
            if y == pc.NONE:
                # evaluator chỉ dùng cột 'after' (và 'before' của câu 1); giữ vị trí hệ thống đã dự đoán nếu có
                if want not in (None, "") and (want == pc.NO_LOC) == (cur == pc.NO_LOC):
                    cur = want
                before = after = cur
            elif y == pc.CREATE:
                before = "-"
                after = want if want not in (None, "", "-") else "?"
            elif y == pc.DESTROY:
                before = cur if cur != "-" else "?"
                after = "-"
            else:  # MOVE
                before = cur if cur != "-" else "?"
                after = want if want not in (None, "", "-") else "?"
            rows.append((rec["paragraph_id"], t, ent, y, before, after))
            cur = after
    return rows


def write_action_file(path: str, rows: Iterable[tuple]) -> None:
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        for r in rows:
            f.write("\t".join(str(x) for x in r) + "\n")


def official_q3(evaluator_dir: str, answers_path: str, predictions_path: str) -> Dict[str, object]:
    """Gọi đúng chuỗi hàm trong evaluator.main() của mã gốc (không sửa mã gốc), trả về P/R/F1
    từng câu hỏi + điểm từng đoạn (để bootstrap). Lỗi định dạng của evaluator (sys.exit(2))
    được đổi thành PredictionError kèm thông báo gốc."""
    import contextlib
    import importlib
    import sys
    if evaluator_dir not in sys.path:
        sys.path.insert(0, evaluator_dir)
    ActionFile = importlib.import_module("process").ActionFile
    QuestionScores = importlib.import_module("scoring").QuestionScores
    Evaluation = importlib.import_module("evaluation").Evaluation
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            predictions = ActionFile.from_file(predictions_path)
            answers = ActionFile.from_file(answers_path)
            diff = answers.diff_participants(predictions)
            if diff:
                raise PredictionError("Thực thể không khớp evaluator: " + "; ".join(diff[:5]))
            ps, asum = predictions.summarize(), answers.summarize()
            scores = {}
            for pid, a in asum.items():
                if pid not in ps:
                    raise PredictionError(f"Evaluator: thiếu dự đoán cho đoạn {pid}")
                scores[pid] = QuestionScores.from_summaries(a, ps[pid])
            ev = Evaluation(scores)
    except SystemExit as e:
        raise PredictionError(f"Evaluator gốc từ chối file (exit {e.code}): {buf.getvalue().strip()}")
    out = {}
    for q in ("inputs", "outputs", "conversions", "moves", "overall"):
        m = getattr(ev, q)
        out[q] = {"P": m.precision, "R": m.recall, "F1": m.F1()}
    out["overall_rounded"] = {"precision": round(ev.overall.precision, 3), "recall": round(ev.overall.recall, 3),
                              "f1": round(ev.overall.F1(), 3)}
    out["_per_paragraph"] = {str(pid): {q: (getattr(s, q).precision, getattr(s, q).recall)
                                        for q in ("inputs", "outputs", "conversions", "moves")}
                             for pid, s in scores.items()}
    return out


# ----------------------------------------------------------------------------- bootstrap theo đoạn
def _f1_from(tp, npred, ngold):
    return prf(tp, npred, ngold)["F1"]


def paired_bootstrap_pair_f1(res_a: dict, res_b: dict, slice_name: str = "all", n_boot: int = 10000,
                             seed: int = 2026) -> Dict[str, float]:
    """CI 95% của ΔF1 = F1(A) − F1(B), lấy mẫu lặp lại THEO ĐOẠN, cùng mẫu cho A và B."""
    pa, pb = res_a["_per_paragraph"], res_b["_per_paragraph"]
    pids = sorted(set(pa) & set(pb))
    if set(pa) != set(pb):
        raise ValueError("Hai hệ thống phải được chấm trên cùng tập đoạn")
    A = [pa[p]["pairs"][slice_name] for p in pids]
    B = [pb[p]["pairs"][slice_name] for p in pids]
    rng = random.Random(seed)
    N = len(pids)

    def f1(arr, idx):
        return _f1_from(sum(arr[i][0] for i in idx), sum(arr[i][1] for i in idx), sum(arr[i][2] for i in idx))

    full = list(range(N))
    delta = f1(A, full) - f1(B, full)
    ds = []
    for _ in range(n_boot):
        idx = [rng.randrange(N) for _ in range(N)]
        ds.append(f1(A, idx) - f1(B, idx))
    ds.sort()
    lo, hi = ds[int(0.025 * n_boot)], ds[int(0.975 * n_boot) - 1]
    p_le0 = sum(d <= 0 for d in ds) / n_boot
    return {"delta": delta, "ci_low": lo, "ci_high": hi, "p_boot_delta_le_0": p_le0, "n_paragraphs": N,
            "n_boot": n_boot, "slice": slice_name}


def bootstrap_ci_pair_f1(res: dict, slice_name: str = "all", n_boot: int = 10000, seed: int = 2026):
    pp = res["_per_paragraph"]
    pids = sorted(pp)
    A = [pp[p]["pairs"][slice_name] for p in pids]
    rng = random.Random(seed)
    N = len(pids)
    vals = []
    for _ in range(n_boot):
        idx = [rng.randrange(N) for _ in range(N)]
        vals.append(_f1_from(sum(A[i][0] for i in idx), sum(A[i][1] for i in idx), sum(A[i][2] for i in idx)))
    vals.sort()
    return {"ci_low": vals[int(0.025 * n_boot)], "ci_high": vals[int(0.975 * n_boot) - 1]}


def mcnemar_exact(a_correct: List[bool], b_correct: List[bool]) -> Dict[str, float]:
    """McNemar chính xác (hai phía) cho hai bộ chọn trên CÙNG các điểm quyết định."""
    b = sum(x and not y for x, y in zip(a_correct, b_correct))
    c = sum(y and not x for x, y in zip(a_correct, b_correct))
    n = b + c
    if n == 0:
        return {"b": 0, "c": 0, "p": 1.0}
    k = min(b, c)
    p = sum(math.comb(n, j) for j in range(k + 1)) / 2 ** n
    return {"b": b, "c": c, "p": min(1.0, 2 * p)}


# ----------------------------------------------------------------------------- đọc action file của hệ thống khác
def action_rows_to_prediction(rows: List[tuple], rec: dict) -> dict:
    """Đổi action file (ví dụ dự đoán ProStruct/ProGlobal công bố) sang định dạng dự đoán chuẩn.
    Ghép theo (câu, tên thực thể); thực thể trùng tên được gán theo thứ tự xuất hiện."""
    n, T = len(rec["entities"]), len(rec["sentences"])
    by_step = defaultdict(list)
    for r in rows:
        if str(r[0]) == rec["paragraph_id"]:
            by_step[int(r[1])].append(r)
    labels = [[pc.NONE] * T for _ in range(n)]
    loc_after = [["?"] * T for _ in range(n)]
    loc_init = ["?"] * n
    for t in range(1, T + 1):
        used = set()
        for r in by_step.get(t, []):
            cand = [i for i, e in enumerate(rec["entities"]) if e == r[2] and i not in used]
            if not cand:
                raise PredictionError(f"{rec['paragraph_id']}: thực thể lạ {r[2]!r} ở câu {t}")
            i = cand[0]
            used.add(i)
            labels[i][t - 1] = r[3]
            before, after = (r[4] or "?"), (r[5] or "?")
            loc_after[i][t - 1] = after
            if t == 1:
                loc_init[i] = before
    exists_initial = [loc_init[i] != pc.NO_LOC if any(by_step.values()) else True for i in range(n)]
    return {"paragraph_id": rec["paragraph_id"], "labels": labels, "exists_initial": exists_initial,
            "location_initial": loc_init, "locations_after": loc_after}
'''
MODULES['propara_metrics_tests.py'] = r'''
"""Kiểm thử bộ chấm (PIPELINE §6.5) — chạy bằng pytest hoặc run_all().

Ví dụ tự dựng: đoạn 2 câu, 4 thực thể.
  câu 1: 'Water and carbon dioxide enter the leaf.'  -> không đổi tồn tại
  câu 2: 'Sugar is made.'  -> water, carbon dioxide bị huỷ (NGẦM); sugar được tạo (được nhắc)
"""
import random

import propara_core as pc
import propara_metrics as pm


def _rec():
    raw = {"para_id": "t1",
           "sentence_texts": ["Water and carbon dioxide enter the leaf.", "Sugar is made."],
           "participants": ["water", "carbon dioxide", "sugar", "oxygen"],
           "states": [["root", "leaf", "-"], ["air", "leaf", "-"], ["-", "-", "leaf"], ["-", "-", "-"]]}
    return pc.build_record(raw, "dev")


def _pred(rec, labels=None, pairs=None):
    p = pm.gold_as_prediction(rec)
    if labels is not None:
        p["labels"] = labels
    if pairs is not None:
        p["pairs"] = pairs
    return p


def _score(rec, pred):
    return pm.evaluate({rec["paragraph_id"]: pred}, [rec])


def test_perfect_prediction_f1_is_one():
    rec = _rec()
    r = _score(rec, pm.gold_as_prediction(rec))
    assert r["PAIR@all"]["F1"] == 1.0 and r["PAIR@all"]["n_gold"] == 2
    assert r["PAIR@src_implicit"]["F1"] == 1.0 and r["PAIR@src_implicit"]["n_gold"] == 2
    assert r["VIOLATION_RATE_per_entity"] == 0 and r["EXIST_ACC"] == 1.0 and r["PAIR_LOC"]["F1"] == 1.0


def test_no_correct_pair_gives_zero():
    rec = _rec()
    r = _score(rec, pm.all_none_prediction(rec))
    assert r["PAIR@all"]["F1"] == 0.0 and r["PAIR@all"]["n_pred"] == 0 and r["PAIR@all"]["R"] == 0.0


def test_swapped_direction_is_wrong():
    rec = _rec()
    r = _score(rec, _pred(rec, pairs=[{"sentence_id": 2, "source_idx": 2, "target_idx": 0}]))
    assert r["PAIR@all"]["tp"] == 0 and r["PAIR@all"]["n_pred"] == 1


def test_wrong_sentence_is_wrong():
    rec = _rec()
    r = _score(rec, _pred(rec, pairs=[{"sentence_id": 1, "source_idx": 0, "target_idx": 2}]))
    assert r["PAIR@all"]["tp"] == 0


def test_correct_pair_wrong_location_only_hurts_pair_loc():
    rec = _rec()
    p = pm.gold_as_prediction(rec)
    p["locations_after"][2] = ["-", "root"]          # sugar tạo ở 'root' thay vì 'leaf'
    r = _score(rec, p)
    assert r["PAIR@all"]["F1"] == 1.0 and r["PAIR_LOC"]["tp"] == 0


def test_duplicate_pairs_counted_once_and_empty_sets_safe():
    rec = _rec()
    dup = [{"sentence_id": 2, "source_idx": 0, "target_idx": 2}] * 3
    r = _score(rec, _pred(rec, pairs=dup))
    assert r["PAIR@all"]["n_pred"] == 1 and r["PAIR@all"]["P"] == 1.0 and r["PAIR@all"]["R"] == 0.5
    assert pm.prf(0, 0, 0)["F1"] == 0.0          # không chia cho 0


def test_destroy_twice_without_create_is_violation():
    rec = _rec()
    labels = [list(r) for r in rec["labels"]]
    labels[0] = [pc.DESTROY, pc.DESTROY]
    p = _pred(rec, labels=labels)
    p["exists_initial"] = [True, True, False, False]
    r = _score(rec, p)
    assert r["VIOLATION_RATE_per_entity"] == 1 / 4


def test_unknown_entity_index_raises():
    rec = _rec()
    try:
        _score(rec, _pred(rec, pairs=[{"sentence_id": 2, "source_idx": 0, "target_idx": 9}]))
    except pm.PredictionError:
        return
    raise AssertionError("phải báo PredictionError")


def test_bad_label_and_shape_raise():
    rec = _rec()
    for bad in ([["NONE", "BURN"]] + [["NONE", "NONE"]] * 3, [["NONE"]] * 4):
        try:
            _score(rec, _pred(rec, labels=bad))
        except pm.PredictionError:
            continue
        raise AssertionError("phải báo PredictionError")


def test_scoring_is_deterministic():
    rec = _rec()
    p = _pred(rec, pairs=[{"sentence_id": 2, "source_idx": 1, "target_idx": 2}])
    a, b = _score(rec, p), _score(rec, p)
    assert a["PAIR@all"] == b["PAIR@all"]


def test_missing_paragraph_strict():
    rec = _rec()
    try:
        pm.evaluate({}, [rec])
    except pm.PredictionError:
        return
    raise AssertionError("thiếu đoạn phải báo lỗi khi strict=True")


def test_slices_use_text_mentions_for_predictions():
    rec = _rec()
    # dự đoán cặp oxygen -> sugar: oxygen không được nhắc ở câu 2 nên thuộc lát src_implicit
    r = _score(rec, _pred(rec, pairs=[{"sentence_id": 2, "source_idx": 3, "target_idx": 2}]))
    assert r["PAIR@src_implicit"]["n_pred"] == 1 and r["PAIR@both_explicit"]["n_pred"] == 0


def test_label_derived_pairs_when_no_pairs_field():
    rec = _rec()
    p = pm.gold_as_prediction(rec)
    assert pm.predicted_pairs(p, rec) == {(2, 0, 2), (2, 1, 2)}


def test_action_rows_are_valid_even_for_violations():
    rec = _rec()
    labels = [[pc.DESTROY, pc.DESTROY], [pc.MOVE, pc.NONE], [pc.CREATE, pc.CREATE], [pc.NONE, pc.NONE]]
    rows = pm.to_official_action_rows({"paragraph_id": "t1", "labels": labels}, rec)
    for _, _, _, y, b, a in rows:
        if y == pc.NONE:
            assert b == a
        if y == pc.CREATE:
            assert b == "-" and a not in ("", "-")
        if y == pc.DESTROY:
            assert b not in ("", "-") and a == "-"
        if y == pc.MOVE:
            assert b not in ("", "-") and a not in ("", "-")


def test_gold_roundtrip_through_action_rows():
    rec = _rec()
    rows = pm.to_official_action_rows(pm.gold_as_prediction(rec), rec)
    assert pc.compare_action_rows(pc.to_action_rows(rec), rows)["n_mismatched_steps"] == 0
    back = pm.action_rows_to_prediction(rows, rec)
    assert back["labels"] == rec["labels"]


def test_paired_bootstrap_identical_systems_zero():
    rec = _rec()
    r = _score(rec, pm.gold_as_prediction(rec))
    bs = pm.paired_bootstrap_pair_f1(r, r, n_boot=200)
    assert bs["delta"] == 0 and bs["ci_low"] == 0 and bs["ci_high"] == 0


def test_mcnemar():
    assert pm.mcnemar_exact([True] * 10, [True] * 10)["p"] == 1.0
    assert pm.mcnemar_exact([True] * 10 + [False] * 0, [False] * 10)["p"] < 0.01


def run_all(verbose=True):
    names = sorted(n for n in globals() if n.startswith("test_"))
    failed = []
    for n in names:
        try:
            globals()[n]()
            if verbose:
                print(f"  PASS  {n}")
        except Exception as e:  # noqa: BLE001
            failed.append((n, repr(e)))
            print(f"  FAIL  {n}: {e!r}")
    return len(names), failed
'''
MODULES['propara_heuristics.py'] = r'''
"""propara_heuristics — các heuristic mốc (mục 2.3 báo cáo) + hệ thống đầu-cuối đơn giản.

Phiên bản: heur_v1. Phụ thuộc propara_core, propara_metrics.
"""
from __future__ import annotations

import random
from typing import Dict, List, Optional

import propara_core as pc

HEUR_VERSION = "heur_v1"


# ----------------------------------------------------------------------------- bộ chọn trên điểm quyết định
def _unmentioned(rec: dict, t: int, cands: List[int]) -> List[int]:
    return [i for i in cands if not rec["mentions"][i][t - 1]]


def last_mention_before(rec: dict, i: int, t: int) -> Optional[int]:
    """Câu gần nhất TRƯỚC câu t có nhắc thực thể i (None nếu chưa từng nhắc)."""
    for k in range(t - 1, 0, -1):
        if rec["mentions"][i][k - 1]:
            return k
    return None


def expected_acc(dp: dict, rec: dict, method: str) -> float:
    """Độ chính xác KỲ VỌNG (tính chính xác, không mô phỏng) của bộ chọn tại một điểm quyết định."""
    U, G = dp["candidates"], set(dp["gold"])
    t = dp["sentence_id"]
    if method == "random":
        pool = U
    elif method == "random_unmentioned":
        pool = _unmentioned(rec, t, U) or U
    elif method == "nearest_mention":
        last = {i: last_mention_before(rec, i, t) for i in U}
        seen = [i for i in U if last[i] is not None]
        if not seen:
            pool = U
        else:
            best = max(last[i] for i in seen)
            pool = [i for i in seen if last[i] == best]       # hoà -> chọn ngẫu nhiên trong nhóm hoà
    else:
        raise ValueError(method)
    return sum(i in G for i in pool) / len(pool)


def simulate_choice(dp: dict, rec: dict, method: str, rng: random.Random) -> int:
    U, t = dp["candidates"], dp["sentence_id"]
    if method == "random":
        return rng.choice(U)
    if method == "random_unmentioned":
        return rng.choice(_unmentioned(rec, t, U) or U)
    if method == "nearest_mention":
        last = {i: last_mention_before(rec, i, t) for i in U}
        seen = [i for i in U if last[i] is not None]
        if not seen:
            return rng.choice(U)
        best = max(last[i] for i in seen)
        return rng.choice([i for i in seen if last[i] == best])
    raise ValueError(method)


def decision_point_table(dps: List[dict], recs_by_pid: Dict[str, dict], methods=("random", "random_unmentioned", "nearest_mention"),
                         seeds: int = 20, seed0: int = 0) -> Dict[str, Dict[str, object]]:
    """Trả về {method: {label: {expected, sim_mean, n, per_dp_correct_seed0}}}."""
    out = {}
    for m in methods:
        out[m] = {}
        for lab in (pc.DESTROY, pc.CREATE):
            sub = [d for d in dps if d["label"] == lab]
            exp = [expected_acc(d, recs_by_pid[d["paragraph_id"]], m) for d in sub]
            sims = []
            first = None
            for s in range(seeds):
                rng = random.Random(seed0 + s)
                corr = [simulate_choice(d, recs_by_pid[d["paragraph_id"]], m, rng) in set(d["gold"]) for d in sub]
                sims.append(sum(corr) / len(corr) if corr else float("nan"))
                if first is None:
                    first = corr
            out[m][lab] = {"n": len(sub), "expected": sum(exp) / len(exp) if exp else float("nan"),
                           "sim_mean_20seeds": sum(sims) / len(sims) if sims else float("nan"),
                           "expected_per_dp": exp}
    return out


# ----------------------------------------------------------------------------- hệ thống đầu-cuối (để chạy toàn bộ luồng chấm)
def all_none(rec: dict) -> dict:
    n, T = len(rec["entities"]), len(rec["sentences"])
    return {"paragraph_id": rec["paragraph_id"], "labels": [[pc.NONE] * T for _ in range(n)],
            "exists_initial": [True] * n}


def oracle_mentioned_only(rec: dict) -> dict:
    """CHẨN ĐOÁN (dùng vàng): giữ nhãn vàng ở ô được nhắc, đặt NONE ở mọi ô ngầm.
    Cho biết hệ thống 'hoàn hảo trên ô được nhắc nhưng mù với thay đổi ngầm' đạt bao nhiêu → trần
    của các phương pháp không suy luận ngầm, và khoảng trống mà đề tài nhắm tới."""
    labels = [[(y if not rec["implicit"][i][t] else pc.NONE) for t, y in enumerate(row)]
              for i, row in enumerate(rec["labels"])]
    return {"paragraph_id": rec["paragraph_id"], "labels": labels,
            "exists_initial": [e[0] for e in rec["existence"]],
            "location_initial": [s[0] for s in rec["states"]],
            "locations_after": [s[1:] for s in rec["states"]]}


def oracle_mentioned_plus_nearest(rec: dict, dps_of_para: List[dict], rng: random.Random) -> dict:
    """CHẨN ĐOÁN (dùng vàng cho ô được nhắc + VỊ TRÍ điểm quyết định): tại mỗi điểm quyết định,
    gán nhãn bằng heuristic nhắc-gần-nhất cho đúng số thực thể vàng ngầm. Đo heuristic ở mức cặp."""
    pred = oracle_mentioned_only(rec)
    for d in dps_of_para:
        U = list(d["candidates"])
        k = len(d["gold"])
        chosen = []
        for _ in range(k):
            dd = dict(d, candidates=[i for i in U if i not in chosen])
            if not dd["candidates"]:
                break
            chosen.append(simulate_choice(dd, rec, "nearest_mention", rng))
        for i in chosen:
            pred["labels"][i][d["sentence_id"] - 1] = d["label"]
    return pred


# ----------------------------------------------------------------------------- B1-E2E: heuristic đầu-cuối KHÔNG dùng vàng
CREATE_CUES = set("""form forms formed forming produce produces produced producing create creates created creating make makes made
making become becomes became develop develops developed generate generates generated grow grows grew release releases released""".split())
DESTROY_CUES = set("""use uses used using consume consumes consumed consuming decompose decomposes decomposed decomposing burn burns
burned burnt burning die dies died dying melt melts melted melting evaporate evaporates evaporated evaporating dissolve dissolves
dissolved eat eats ate eaten break breaks broke broken""".split())
TRANSFORM = set("turn turns turned turning change changes changed convert converts converted become becomes became form forms formed".split())


def rule_nearest_e2e(rec: dict) -> dict:
    """Heuristic đầu-cuối (B1-E2E), luật cố định viết TRƯỚC khi xem điểm dev/test:
    (1) Thực thể ĐƯỢC NHẮC: trước 'into' trong câu có động từ biến đổi → DESTROY, sau 'into' → CREATE;
        nếu không: câu có từ gợi tạo → CREATE, câu có từ gợi huỷ → DESTROY.
    (2) Tồn tại ban đầu = không (sự kiện đầu tiên của thực thể là CREATE).
    (3) Ngầm (heuristic 'nhắc gần nhất' của báo cáo, B1c): câu có CREATE mà không DESTROY → huỷ thực thể đang tồn tại,
        không được nhắc, được nhắc gần nhất trước đó; ngược lại có DESTROY mà không CREATE → tạo thực thể chưa tồn tại,
        không được nhắc, được nhắc gần nhất trước đó. Không có ứng viên từng được nhắc → bỏ qua (không đoán bừa).
    (4) Áp ràng buộc tồn tại khi đi qua các câu."""
    n, T = len(rec["entities"]), len(rec["sentences"])
    words = [pc.tokenize(s) for s in rec["sentences"]]
    raw = [[pc.NONE] * T for _ in range(n)]
    for t in range(1, T + 1):
        ws = words[t - 1]
        has_into = "into" in ws and any(w in TRANSFORM for w in ws)
        into_pos = ws.index("into") if "into" in ws else -1
        cue_c = any(w in CREATE_CUES for w in ws)
        cue_d = any(w in DESTROY_CUES for w in ws)
        for i, ent in enumerate(rec["entities"]):
            if not rec["mentions"][i][t - 1]:
                continue
            pos = [k for k, w in enumerate(ws) if pc.is_mentioned(ent, w)]
            if has_into and pos:
                raw[i][t - 1] = pc.DESTROY if min(pos) < into_pos else pc.CREATE
            elif cue_c and not cue_d:
                raw[i][t - 1] = pc.CREATE
            elif cue_d and not cue_c:
                raw[i][t - 1] = pc.DESTROY
    init = [next((y for y in raw[i] if y != pc.NONE), pc.NONE) != pc.CREATE for i in range(n)]
    labels = [[pc.NONE] * T for _ in range(n)]
    e = list(init)
    for t in range(1, T + 1):
        for i in range(n):
            y = raw[i][t - 1]
            if y in pc.VALID_LABELS[e[i]] and y != pc.NONE:
                labels[i][t - 1] = y
        D = [i for i in range(n) if labels[i][t - 1] == pc.DESTROY]
        C = [i for i in range(n) if labels[i][t - 1] == pc.CREATE]
        def nearest(cands):
            last = {i: last_mention_before(rec, i, t) for i in cands}
            seen = [i for i in cands if last[i] is not None]
            return max(seen, key=lambda i: (last[i], -i)) if seen else None
        if C and not D:
            j = nearest([i for i in range(n) if e[i] and not rec["mentions"][i][t - 1] and labels[i][t - 1] == pc.NONE])
            if j is not None:
                labels[j][t - 1] = pc.DESTROY
        elif D and not C:
            j = nearest([i for i in range(n) if not e[i] and not rec["mentions"][i][t - 1] and labels[i][t - 1] == pc.NONE])
            if j is not None:
                labels[j][t - 1] = pc.CREATE
        for i in range(n):
            if labels[i][t - 1] == pc.CREATE:
                e[i] = True
            elif labels[i][t - 1] == pc.DESTROY:
                e[i] = False
    return {"paragraph_id": rec["paragraph_id"], "labels": labels, "exists_initial": init}


def rule_only_e2e(rec: dict) -> dict:
    """Như rule_nearest_e2e nhưng BỎ bước (3) suy ngầm → đo riêng đóng góp của heuristic ngầm."""
    p = rule_nearest_e2e(rec)
    n, T = len(rec["entities"]), len(rec["sentences"])
    labels = [[(y if rec["mentions"][i][t] else pc.NONE) for t, y in enumerate(p["labels"][i])] for i in range(n)]
    return {"paragraph_id": rec["paragraph_id"], "labels": labels, "exists_initial": p["exists_initial"]}
'''
MODULES['propara_pp3.py'] = r'''
"""propara_pp3 — PP3: khai thác cặp chuyển đổi X→Y từ WikiHow bằng mẫu cố định + bộ chấm có hướng E(X,Y) (mục 4.5).

Phiên bản: pp3_v1. Phần trích mẫu chỉ dùng thư viện chuẩn; phần huấn luyện cần torch.

Rút gọn so với báo cáo (ghi rõ để không nhầm): báo cáo rút cụm về "danh từ đầu ở dạng gốc"; ở đây
KHÔNG có bộ phân tích cú pháp, danh từ đầu = từ cuối của cụm sau khi cắt ở từ ranh giới, đưa về số ít
bằng luật đơn giản. Các mẫu được KHOÁ trước khi chạy (PATTERNS_V1).
"""
from __future__ import annotations

import math
import random
import re
from collections import Counter, defaultdict
from typing import Callable, Dict, Iterable, List, Optional, Tuple

import propara_core as pc

PP3_VERSION = "pp3_v2"

# ----------------------------------------------------------------------------- mẫu (khoá trước)
_V_TURN_T = r"(?:turn|turns|turned|turning|convert|converts|converted|converting|transform|transforms|transformed|transforming)"
_V_INTR = (r"(?:turns|turned|turn|turning|changes|changed|change|changing|breaks down|broke down|break down|breaking down|"
           r"decomposes|decomposed|decompose|decomposing|melts|melted|melt|melting)")
_V_PASS = r"(?:is|are|was|were|be|been|being|gets|get|got|getting)\s+(?:turned|converted|transformed)"
_V_MAKE = r"(?:make|makes|made|making)"
_BE = r"(?:is|are|was|were|be|been|being|can be|will be)"

# Mỗi mẫu: (tên, regex với nhóm có tên L (bên trái), R (bên phải), hướng): hướng "LR" nghĩa là X=L, Y=R.
PATTERNS_V1 = [
    ("turn_X_into_Y", re.compile(rf"\b{_V_TURN_T}\s+(?P<L>[^.,;:!?]+?)\s+into\s+(?P<R>[^.,;:!?]+)"), "LR"),
    ("X_turns_into_Y", re.compile(rf"(?P<L>[^.,;:!?]+?)\s+{_V_INTR}\s+into\s+(?P<R>[^.,;:!?]+)"), "LR"),
    ("X_is_turned_into_Y", re.compile(rf"(?P<L>[^.,;:!?]+?)\s+{_V_PASS}\s+into\s+(?P<R>[^.,;:!?]+)"), "LR"),
    ("make_Y_from_X", re.compile(rf"\b{_V_MAKE}\s+(?P<L>[^.,;:!?]+?)\s+(?:from|out of)\s+(?P<R>[^.,;:!?]+)"), "RL"),
    ("Y_is_made_from_X", re.compile(rf"(?P<L>[^.,;:!?]+?)\s+{_BE}\s+made\s+(?:from|out of)\s+(?P<R>[^.,;:!?]+)"), "RL"),
    ("Y_is_formed_from_X", re.compile(rf"(?P<L>[^.,;:!?]+?)\s+{_BE}\s+formed\s+from\s+(?P<R>[^.,;:!?]+)"), "RL"),
    ("Y_forms_from_X", re.compile(rf"(?P<L>[^.,;:!?]+?)\s+(?:forms|form|formed|forming)\s+from\s+(?P<R>[^.,;:!?]+)"), "RL"),
]

# Từ ranh giới: cụm bên phải bị cắt TRƯỚC từ đầu tiên thuộc tập này; cụm bên trái chỉ giữ phần SAU từ cuối thuộc tập này.
BOUNDARY = set("""when if and or but by with in on at for to that which who because while so as using until
after before then than of about over under through during without within into onto from is are was were be been
can will would should could may might must do does did has have had not you your yourself it they them this these
those we i he she its their his her our my me us also just simply easily quickly slowly""".split())
BAD_HEADS = set("""it them this that these those one ones thing things something anything everything stuff way ways
you yourself he she they we i what which who whatever part parts piece pieces bit bits kind type lot lots amount
amounts number numbers form forms shape shapes use uses order sure time times step steps sense good bad""".split())


IRREGULAR = {"leaves": "leaf", "halves": "half", "loaves": "loaf", "shelves": "shelf", "calves": "calf", "wolves": "wolf",
             "knives": "knife", "lives": "life", "wives": "wife", "mice": "mouse", "geese": "goose", "teeth": "tooth",
             "feet": "foot", "children": "child", "people": "person", "men": "man", "women": "woman", "oxen": "ox"}
QUANTIFIERS = set("""layer layers pile piles bunch bunches cup cups handful handfuls piece pieces bit bits lot lots amount amounts
drop drops bag bags sheet sheets slice slices kind kinds type types sort sorts mixture""".split())


NO_SINGULAR = {"canvas", "gas", "atlas", "lens", "species", "series", "news", "physics", "mathematics", "chaos", "bias", "alias"}


def singularize(w: str) -> str:
    if w in IRREGULAR:
        return IRREGULAR[w]
    if w in NO_SINGULAR:
        return w
    if len(w) <= 3:
        return w
    if w.endswith("ies") and len(w) > 4:
        return w[:-3] + "y"
    if w.endswith(("sses", "shes", "ches", "xes", "zes")):
        return w[:-2]
    if w.endswith("s") and not w.endswith(("ss", "us", "is")):
        return w[:-1]
    return w


def _words(s: str) -> List[str]:
    return re.findall(r"[a-z][a-z\-']*", s.lower())


def head_right(phrase: str, max_words: int = 5) -> Optional[str]:
    """Cụm bên phải: cắt ở từ ranh giới đầu tiên, lấy từ cuối còn lại."""
    out = []
    ws = _words(phrase)
    while len(ws) >= 3 and ws[0] in ("a", "an", "the") and ws[1] in QUANTIFIERS and ws[2] == "of":
        ws = ws[3:]
    while len(ws) >= 2 and ws[0] in QUANTIFIERS and ws[1] == "of":       # 'layers of sediment' -> sediment
        ws = ws[2:]
    for w in ws[:max_words + 3]:
        if w in BOUNDARY and out:
            break
        if w in ("a", "an", "the", "some", "any", "more", "small", "large", "new"):
            continue
        out.append(w)
        if len(out) >= max_words:
            break
    return _clean_head(out[-1]) if out else None


def head_left(phrase: str) -> Optional[str]:
    """Cụm bên trái: chỉ giữ phần sau từ ranh giới cuối cùng, lấy từ cuối (sát động từ)."""
    ws = _words(phrase)
    keep = []
    for w in reversed(ws):
        if w in BOUNDARY:
            break
        keep.append(w)
        if len(keep) >= 5:
            break
    keep = [w for w in reversed(keep) if w not in ("a", "an", "the", "some", "any")]
    return _clean_head(keep[-1]) if keep else None


def _clean_head(w: str) -> Optional[str]:
    w = w.strip("-'").replace("'s", "")
    w = singularize(w)
    if len(w) < 3 or w in BAD_HEADS or w in BOUNDARY:
        return None
    return w


def extract_pairs_from_sentence(sentence: str) -> List[Tuple[str, str, str]]:
    """→ [(X, Y, tên_mẫu)] với X→Y."""
    s = sentence.strip()
    out = []
    for name, rx, direction in PATTERNS_V1:
        for mt in rx.finditer(s.lower()):
            # cụm L là tân ngữ đứng SAU động từ (turn X into, make Y from) hoặc chủ ngữ đứng TRƯỚC động từ (các mẫu còn lại)
            L = head_right(mt.group("L")) if name in ("turn_X_into_Y", "make_Y_from_X") else head_left(mt.group("L"))
            R = head_right(mt.group("R"))
            if not L or not R or L == R:
                continue
            X, Y = (L, R) if direction == "LR" else (R, L)
            out.append((X, Y, name))
    return out


# ----------------------------------------------------------------------------- pp3_v2: trích theo TỪ LOẠI (POS)
# Cùng 7 mẫu của báo cáo, nhưng danh từ đầu phải được gán nhãn NN*/NNS (bộ gán nhãn perceptron của NLTK).
TURN_T = {"turn", "turns", "turned", "turning", "convert", "converts", "converted", "converting",
          "transform", "transforms", "transformed", "transforming"}
INTR_1 = {"turns", "turned", "turn", "turning", "changes", "changed", "change", "changing", "decomposes", "decomposed",
          "decompose", "decomposing", "melts", "melted", "melt", "melting"}
BREAK = {"breaks", "broke", "break", "breaking"}
BE_GET = {"is", "are", "was", "were", "be", "been", "being", "gets", "get", "got", "getting"}
PASS_V = {"turned", "converted", "transformed"}
MAKE = {"make", "makes", "made", "making"}
FORM_V = {"forms", "form", "formed", "forming"}
NOUN_TAGS = {"NN", "NNS", "NNP", "NNPS"}
NP_TAGS = NOUN_TAGS | {"DT", "PDT", "PRP$", "JJ", "JJR", "JJS", "CD", "POS", "VBN", "VBG"}
_TOK = re.compile(r"[A-Za-z][A-Za-z\-']*|[.,;:!?()]")


def _np_after(toks, tags, p):
    """Cụm danh từ bắt đầu tại p; trả về (danh từ đầu, vị trí kết thúc) hoặc (None, p)."""
    j, head, hpos = p, None, None
    while j < len(toks) and tags[j] in NP_TAGS:
        if tags[j] in NOUN_TAGS:
            head, hpos = toks[j], j
        elif head is not None and tags[j] not in NOUN_TAGS:
            break
        j += 1
    if head is not None and head.lower() in QUANTIFIERS and hpos + 1 < len(toks) and toks[hpos + 1].lower() == "of":
        return _np_after(toks, tags, hpos + 2)
    return head, j


def _np_before(toks, tags, e):
    """Danh từ đầu của cụm kết thúc tại e (ngay trước động từ); phải là danh từ."""
    if e < 0 or tags[e] not in NOUN_TAGS:
        return None
    return toks[e]


def _norm_head(w):
    return _clean_head(w.lower()) if w else None


def extract_pairs_pos(sentence: str, tagger) -> List[Tuple[str, str, str]]:
    toks = _TOK.findall(sentence)
    if len(toks) < 4:
        return []
    tags = [t for _, t in tagger(toks)]
    low = [t.lower() for t in toks]
    out = []
    n = len(toks)
    for i, w in enumerate(low):
        try:
            if w in TURN_T and i + 1 < n and low[i + 1] != "into":
                obj, j = _np_after(toks, tags, i + 1)
                if obj and j < n and low[j] == "into":
                    tgt, _ = _np_after(toks, tags, j + 1)
                    out.append((_norm_head(obj), _norm_head(tgt), "turn_X_into_Y"))
            if (w in INTR_1 and i + 1 < n and low[i + 1] == "into") or (w in BREAK and i + 2 < n and low[i + 1] == "down" and low[i + 2] == "into"):
                k = i + 1 if w in INTR_1 else i + 2
                if i > 0 and low[i - 1] in BE_GET:
                    continue                                    # 'is turned into' xử lý ở mẫu bị động
                src = _np_before(toks, tags, i - 1)
                tgt, _ = _np_after(toks, tags, k + 1)
                out.append((_norm_head(src), _norm_head(tgt), "X_turns_into_Y"))
            if w in BE_GET and i + 2 < n and low[i + 1] in PASS_V and low[i + 2] == "into":
                src = _np_before(toks, tags, i - 1)
                tgt, _ = _np_after(toks, tags, i + 3)
                out.append((_norm_head(src), _norm_head(tgt), "X_is_turned_into_Y"))
            if w in MAKE and not (i > 0 and low[i - 1] in BE_GET):
                obj, j = _np_after(toks, tags, i + 1)
                if obj and j < n and (low[j] == "from" or (low[j] == "out" and j + 1 < n and low[j + 1] == "of")):
                    src, _ = _np_after(toks, tags, j + (1 if low[j] == "from" else 2))
                    out.append((_norm_head(src), _norm_head(obj), "make_Y_from_X"))
            if w in BE_GET and i + 2 < n and low[i + 1] == "made" and (low[i + 2] == "from" or (low[i + 2] == "out" and i + 3 < n and low[i + 3] == "of")):
                tgt = _np_before(toks, tags, i - 1)
                src, _ = _np_after(toks, tags, i + (3 if low[i + 2] == "from" else 4))
                out.append((_norm_head(src), _norm_head(tgt), "Y_is_made_from_X"))
            if w in BE_GET and i + 2 < n and low[i + 1] == "formed" and low[i + 2] == "from":
                tgt = _np_before(toks, tags, i - 1)
                src, _ = _np_after(toks, tags, i + 3)
                out.append((_norm_head(src), _norm_head(tgt), "Y_is_formed_from_X"))
            if w in FORM_V and i + 1 < n and low[i + 1] == "from" and not (i > 0 and low[i - 1] in BE_GET):
                tgt = _np_before(toks, tags, i - 1)
                src, _ = _np_after(toks, tags, i + 2)
                out.append((_norm_head(src), _norm_head(tgt), "Y_forms_from_X"))
        except IndexError:
            continue
    return [(x, y, nm) for x, y, nm in out if x and y and x != y]


_TRIGGER = re.compile(r"\b(?:into|from|out of)\b")
_VERB_HINT = re.compile(r"\b(?:turn|convert|transform|chang|break|broke|decompos|melt|made|make|form)")


_SENT_SPLIT = re.compile(r"(?<=[.!?;])\s+|\n+")


def split_sentences(text: str) -> List[str]:
    return [s for s in _SENT_SPLIT.split(text or "") if len(s) > 5]


def article_key(title: str) -> str:
    return re.sub(r"\d+$", "", (title or "").strip()).lower()


def mine_corpus(rows: Iterable[Tuple[str, str]], log_every: int = 50000, log=print, tagger=None):
    """rows: (title, text). Trả về (Counter (X,Y)->số lần, {(X,Y)->set bài}, Counter mẫu, thống kê, {bài->set danh từ đầu})."""
    pair_cnt, pair_art, pat_cnt = Counter(), defaultdict(set), Counter()
    art_nouns = defaultdict(set)
    n_rows = n_sent = 0
    for title, text in rows:
        n_rows += 1
        art = article_key(title)
        for s in split_sentences(text):
            n_sent += 1
            low = s.lower()
            if not any(k in low for k in (" into ", " from ", " out of ")):
                continue
            if tagger is not None and not _VERB_HINT.search(low):
                continue
            found = extract_pairs_pos(s, tagger) if tagger is not None else extract_pairs_from_sentence(s)
            for X, Y, name in found:
                pair_cnt[(X, Y)] += 1
                pair_art[(X, Y)].add(art)
                pat_cnt[name] += 1
                art_nouns[art].update((X, Y))
        if log_every and n_rows % log_every == 0:
            log(f"  đã quét {n_rows} dòng, {n_sent} câu, {len(pair_cnt)} cặp khác nhau")
    return pair_cnt, pair_art, pat_cnt, {"rows": n_rows, "sentences": n_sent}, art_nouns


# ----------------------------------------------------------------------------- độ phủ trên ProPara
def entity_heads(entity: str) -> List[str]:
    """Danh từ đầu của mọi tên gọi khác của thực thể ProPara (cùng luật với phía WikiHow)."""
    hs = []
    for a in pc.split_aliases(entity):
        h = head_right(a) or head_left(a)
        if h and h not in hs:
            hs.append(h)
    return hs


def coverage(gold_pairs: List[dict], K: set) -> Dict[str, float]:
    """Công thức (24)–(26): phủ cặp / nguồn / đích + tỉ lệ cặp cùng danh từ đầu (không khai thác được bằng mẫu)."""
    sources = {x for x, _ in K}
    targets = {y for _, y in K}
    n = len(gold_pairs)
    c_pair = c_src = c_tgt = same = 0
    for g in gold_pairs:
        hx, hy = entity_heads(g["source"]), entity_heads(g["target"])
        if set(hx) & set(hy):
            same += 1
        c_pair += any((x, y) in K for x in hx for y in hy)
        c_src += any(x in sources for x in hx)
        c_tgt += any(y in targets for y in hy)
    return {"n_gold_pairs": n, "cover_pair": c_pair / max(n, 1), "cover_source": c_src / max(n, 1),
            "cover_target": c_tgt / max(n, 1), "same_head_pairs": same / max(n, 1)}


# ----------------------------------------------------------------------------- tập huấn luyện E(X,Y)
def build_training_pairs(pair_cnt: Counter, pair_art: dict, art_nouns: dict, val_frac: float = 0.1, min_count: int = 1,
                         seed: int = 2026, max_pos: int = 200000):
    """Dương = cặp khớp mẫu. Âm (ba loại như báo cáo): đích ngẫu nhiên, đảo chiều, cùng xuất hiện trong bài nhưng không khớp mẫu
    (âm GIẢ ĐỊNH, có thể nhiễu). Tách train/val theo BÀI: một cặp thuộc val nếu mọi bài chứa nó đều thuộc val."""
    rng = random.Random(seed)
    K = {p for p, c in pair_cnt.items() if c >= min_count}
    pos = sorted(K, key=lambda p: -pair_cnt[p])[:max_pos]
    is_val_art = lambda a: (pc.stable_hash(f"{seed}:{a}") % 1000) < val_frac * 1000
    targets = [y for _, y in pos]
    data = {"train": [], "val": []}
    for (x, y) in pos:
        split = "val" if all(is_val_art(a) for a in pair_art[(x, y)]) else "train"
        data[split].append((x, y, 1, "pos"))
        for _ in range(5):                                   # đích ngẫu nhiên
            y2 = rng.choice(targets)
            if (x, y2) not in K and y2 != x:
                data[split].append((x, y2, 0, "neg_random")); break
        if (y, x) not in K:                                  # đảo chiều
            data[split].append((y, x, 0, "neg_reverse"))
        arts = sorted(pair_art[(x, y)])
        nouns = sorted(art_nouns.get(rng.choice(arts), set()) - {x, y})
        if nouns:                                            # cùng bài, không khớp mẫu
            z = rng.choice(nouns)
            if (x, z) not in K:
                data[split].append((x, z, 0, "neg_cooccur"))
    return data, K


def train_pair_scorer(data: dict, embed: Callable[[List[str]], "torch.Tensor"], epochs: int = 30, lr: float = 1e-3,
                      wd: float = 1e-4, batch: int = 1024, seed: int = 2026, device: str = "cpu", log=print, patience: int = 5):
    """E(X,Y) = σ(u_X^T W v_Y + a^T u_X + b^T v_Y + c), u = v = embedding cố định (công thức 20–22), mất mát BCE (23).
    Chọn epoch theo loss VALIDATION. Trả về (model, lịch sử)."""
    import torch
    import torch.nn as nn
    torch.manual_seed(seed)
    vocab = sorted({w for sp in data.values() for x, y, _, _ in sp for w in (x, y)})
    idx = {w: i for i, w in enumerate(vocab)}
    emb = embed(vocab).float().to(device)
    emb = emb / emb.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    d = emb.shape[1]

    class Scorer(nn.Module):
        def __init__(self):
            super().__init__()
            self.W = nn.Parameter(torch.zeros(d, d))
            self.a = nn.Linear(d, 1)
            self.b = nn.Linear(d, 1, bias=False)
        def forward(self, u, v):
            return (u @ self.W * v).sum(-1) + self.a(u).squeeze(-1) + self.b(v).squeeze(-1)

    model = Scorer().to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    def tens(split):
        rows = data[split]
        return (torch.tensor([idx[x] for x, _, _, _ in rows], device=device), torch.tensor([idx[y] for _, y, _, _ in rows], device=device),
                torch.tensor([z for _, _, z, _ in rows], dtype=torch.float, device=device), [k for *_, k in rows])
    tr, va = tens("train"), tens("val")
    lossf = nn.BCEWithLogitsLoss()
    hist, best, best_state, bad = [], float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(tr[0]), device=device)
        tot = 0.0
        for s in range(0, len(perm), batch):
            j = perm[s:s + batch]
            loss = lossf(model(emb[tr[0][j]], emb[tr[1][j]]), tr[2][j])
            opt.zero_grad(); loss.backward(); opt.step()
            tot += float(loss) * len(j)
        model.eval()
        with torch.no_grad():
            lv = model(emb[va[0]], emb[va[1]])
            vloss = float(lossf(lv, va[2])) if len(lv) else tot / max(len(perm), 1)   # val rỗng → dùng train loss (ghi chú)
            m = _rank_metrics(lv.cpu(), va[2].cpu(), va[3])
        row = {"epoch": ep, "train_loss": tot / max(len(perm), 1), "val_loss": vloss, **m}
        hist.append(row)
        log(f"  PP3 ep{ep} train={row['train_loss']:.4f} val={vloss:.4f} AUC={m['auc']:.3f} AP={m['ap']:.3f} "
            f"AUC_vs_reverse={m['auc_vs_reverse']:.3f}")
        if vloss < best - 1e-4:
            best, bad = vloss, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, hist


def _auc(pos, neg):
    if not pos or not neg:
        return float("nan")
    allv = sorted([(v, 1) for v in pos] + [(v, 0) for v in neg])
    rank_sum, r = 0.0, 1
    i = 0
    while i < len(allv):
        j = i
        while j < len(allv) and allv[j][0] == allv[i][0]:
            j += 1
        avg = (i + 1 + j) / 2
        rank_sum += avg * sum(1 for k in range(i, j) if allv[k][1] == 1)
        i = j
    return (rank_sum - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))


def _rank_metrics(logits, labels, kinds):
    v = logits.tolist(); z = labels.tolist()
    pos = [a for a, b in zip(v, z) if b == 1]
    neg = [a for a, b in zip(v, z) if b == 0]
    rev = [a for a, k in zip(v, kinds) if k == "neg_reverse"]
    order = sorted(range(len(v)), key=lambda i: -v[i])
    hits, ap = 0, 0.0
    for r, i in enumerate(order, 1):
        if z[i] == 1:
            hits += 1
            ap += hits / r
    return {"auc": _auc(pos, neg), "ap": ap / max(hits, 1), "auc_vs_reverse": _auc(pos, rev), "n_val": len(v)}


def logE_matrix(model, embed, rec: dict, device: str = "cpu", floor: float = -10.0) -> List[List[float]]:
    """log E(X_i, X_j) cho mọi cặp thực thể của một đoạn (lấy max trên các tên gọi khác)."""
    import torch
    import torch.nn.functional as F
    n = len(rec["entities"])
    names = [entity_heads(e) or [e.lower()] for e in rec["entities"]]
    flat = sorted({h for hs in names for h in hs})
    E = embed(flat).float().to(device)
    E = E / E.norm(dim=-1, keepdim=True).clamp_min(1e-6)
    pos = {h: k for k, h in enumerate(flat)}
    out = [[0.0] * n for _ in range(n)]
    with torch.no_grad():
        for i in range(n):
            for j in range(n):
                if i == j:
                    continue
                best = floor
                for hx in names[i]:
                    for hy in names[j]:
                        s = model(E[pos[hx]][None], E[pos[hy]][None])
                        best = max(best, float(F.logsigmoid(s)))
                out[i][j] = max(best, floor)
    return out
'''
MODULES['propara_pp4.py'] = r'''
"""propara_pp4 — PP4: RoBERTa + BiLSTM, điểm đơn φ, điểm cặp ψ, CRF giải mã chung (mục 4.3–4.4 báo cáo).

Phiên bản: pp4_v1. Cần torch; phần mô hình cần transformers (nạp muộn).

CRF chính xác trên không gian TỒN TẠI: trạng thái e ∈ {0,1}^n (2^n ≤ 256 trên ProPara vì n ≤ 8).
Chuyển e → e' xác định loại của từng thực thể:
   (0,0) chưa tồn tại, giữ nguyên → NONE
   (0,1) CREATE
   (1,0) DESTROY
   (1,1) đang tồn tại, giữ nguyên → MOVE hoặc NONE (cộng gộp bằng logsumexp, Viterbi lấy max)
Mọi chuỗi ngoài ràng buộc (15) có xác suất 0 theo cấu tạo. Cặp ψ_t(i,j) cộng vào khi i bị DESTROY và j được CREATE
ở cùng câu t (công thức 14); tồn tại của i trước câu và không tồn tại của j trước câu đã được ràng buộc đảm bảo,
nên bộ lọc PP2 không loại thêm cặp nào trong giải mã chính xác (PP2 chỉ còn vai trò tăng tốc/khi giải mã xấp xỉ).
"""
from __future__ import annotations

import math
import os
import random
import re
import time
from typing import Dict, List, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F

import propara_core as pc

PP4_VERSION = "pp4_v1.3"
LAB_IDX = {pc.CREATE: 0, pc.DESTROY: 1, pc.MOVE: 2, pc.NONE: 3}
IDX_LAB = {v: k for k, v in LAB_IDX.items()}
_STATE_CACHE: Dict[Tuple[int, str], Tuple[torch.Tensor, torch.Tensor]] = {}


# ============================================================================ CRF
def _state_tables(n: int, device) -> Tuple[torch.Tensor, torch.Tensor]:
    """bits (S,n) và loại chuyển K (S,S,n) ∈ {0:(0,0),1:(0,1)=C,2:(1,0)=D,3:(1,1)}."""
    key = (n, str(device))
    if key not in _STATE_CACHE:
        S = 1 << n
        idx = torch.arange(S, device=device)
        bits = ((idx[:, None] >> torch.arange(n, device=device)[None, :]) & 1).long()
        K = 2 * bits[:, None, :] + bits[None, :, :]
        _STATE_CACHE[key] = (bits, K)
    return _STATE_CACHE[key]


def _transition_scores(phi_t: torch.Tensor, psi_t: Optional[torch.Tensor], K: torch.Tensor, use_max: bool):
    """phi_t (n,4) theo thứ tự [C,D,M,N]; psi_t (n,n) hoặc None. Trả về (S,S) và (nếu use_max) lựa chọn M/N (n,)."""
    stay = torch.stack([phi_t[:, 2], phi_t[:, 3]], -1)
    stay_val = stay.max(-1).values if use_max else torch.logsumexp(stay, -1)
    table = torch.stack([phi_t[:, 3], phi_t[:, 0], phi_t[:, 1], stay_val], -1)       # (n,4) theo loại 0..3
    n = phi_t.shape[0]
    sc = table[torch.arange(n, device=phi_t.device)[None, None, :].expand_as(K), K].sum(-1)   # (S,S)
    if psi_t is not None:
        Dm = (K == 2).to(phi_t.dtype)
        Cm = (K == 1).to(phi_t.dtype)
        psi0 = psi_t * (1 - torch.eye(n, device=phi_t.device, dtype=phi_t.dtype))
        sc = sc + torch.einsum("abi,ij,abj->ab", Dm, psi0, Cm)
    return sc, (stay[:, 0] >= stay[:, 1])


def crf_log_partition(phi: torch.Tensor, psi: Optional[torch.Tensor], init: torch.Tensor) -> torch.Tensor:
    """phi (T,n,4), psi (T,n,n) hoặc None, init (n,) điểm 'tồn tại ban đầu'. Trả về log Z."""
    T, n, _ = phi.shape
    bits, K = _state_tables(n, phi.device)
    alpha = (bits.to(phi.dtype) * init[None, :]).sum(-1)                               # (S,)
    for t in range(T):
        tr, _ = _transition_scores(phi[t], None if psi is None else psi[t], K, use_max=False)
        alpha = torch.logsumexp(alpha[:, None] + tr, dim=0)
    return torch.logsumexp(alpha, 0)


def crf_existence_marginals(phi: torch.Tensor, psi: Optional[torch.Tensor], init: torch.Tensor) -> torch.Tensor:
    """p(e_{t,i} = 1) cho t = 0..T (t = 0: trước câu 1) bằng forward-backward trên 2^n trạng thái tồn tại.
    Dùng cho PP2 (công thức 11): p(e_{X,t-1}=1) > τ1 và p(e_{Y,t-1}=1) < τ2."""
    with torch.no_grad():
        T, n, _ = phi.shape
        bits, K = _state_tables(n, phi.device)
        trs = [_transition_scores(phi[t], None if psi is None else psi[t], K, use_max=False)[0] for t in range(T)]
        alphas = [(bits.to(phi.dtype) * init[None, :]).sum(-1)]
        for t in range(T):
            alphas.append(torch.logsumexp(alphas[-1][:, None] + trs[t], dim=0))
        logZ = torch.logsumexp(alphas[-1], 0)
        betas = [torch.zeros_like(alphas[0])]
        for t in range(T - 1, -1, -1):
            betas.append(torch.logsumexp(trs[t] + betas[-1][None, :], dim=1))
        betas.reverse()
        out = []
        for t in range(T + 1):
            pst = torch.exp(alphas[t] + betas[t] - logZ)                      # (S,)
            out.append((pst[:, None] * bits.to(phi.dtype)).sum(0))
        return torch.stack(out)                                               # (T+1, n)


def crf_gold_score(phi, psi, init, gold_lab: torch.Tensor, gold_init: torch.Tensor) -> torch.Tensor:
    """gold_lab (T,n) chỉ số nhãn [C,D,M,N]; gold_init (n,) 0/1."""
    T, n, _ = phi.shape
    s = (gold_init.to(phi.dtype) * init).sum()
    s = s + phi.gather(-1, gold_lab[..., None]).squeeze(-1).sum()
    if psi is not None:
        D = (gold_lab == LAB_IDX[pc.DESTROY]).to(phi.dtype)
        C = (gold_lab == LAB_IDX[pc.CREATE]).to(phi.dtype)
        eye = torch.eye(n, device=phi.device, dtype=phi.dtype)
        s = s + torch.einsum("ti,tij,tj->", D, psi * (1 - eye), C)
    return s


def crf_nll(phi, psi, init, gold_lab, gold_init) -> torch.Tensor:
    return crf_log_partition(phi, psi, init) - crf_gold_score(phi, psi, init, gold_lab, gold_init)


def crf_viterbi(phi: torch.Tensor, psi: Optional[torch.Tensor], init: torch.Tensor) -> Tuple[List[List[str]], List[bool], float]:
    """Trả về (nhãn [n][T], tồn tại ban đầu [n], điểm tốt nhất)."""
    with torch.no_grad():
        T, n, _ = phi.shape
        bits, K = _state_tables(n, phi.device)
        delta = (bits.to(phi.dtype) * init[None, :]).sum(-1)
        back, mn_choice = [], []
        for t in range(T):
            tr, mchoice = _transition_scores(phi[t], None if psi is None else psi[t], K, use_max=True)
            cand = delta[:, None] + tr
            delta, arg = cand.max(0)
            back.append(arg)
            mn_choice.append(mchoice)
        e = int(delta.argmax())
        best = float(delta[e])
        path = [e]
        for t in range(T - 1, -1, -1):
            e = int(back[t][e])
            path.append(e)
        path.reverse()                                                                  # e_0 .. e_T
        labels = [[pc.NONE] * T for _ in range(n)]
        for t in range(T):
            a, b = path[t], path[t + 1]
            for i in range(n):
                ea, eb = (a >> i) & 1, (b >> i) & 1
                if ea == 0 and eb == 1:
                    labels[i][t] = pc.CREATE
                elif ea == 1 and eb == 0:
                    labels[i][t] = pc.DESTROY
                elif ea == 1 and eb == 1:
                    labels[i][t] = pc.MOVE if bool(mn_choice[t][i]) else pc.NONE
        init_exists = [bool((path[0] >> i) & 1) for i in range(n)]
        return labels, init_exists, best


def brute_force(phi, psi, init):
    """Liệt kê vét cạn mọi cấu hình hợp lệ (chỉ dùng trong kiểm thử, n·T rất nhỏ)."""
    import itertools
    T, n, _ = phi.shape
    best, best_cfg, scores = -float("inf"), None, []
    for e0 in itertools.product([0, 1], repeat=n):
        per_entity_opts = []
        for i in range(n):
            seqs = []
            def rec(t, e, acc):
                if t == T:
                    seqs.append(tuple(acc)); return
                for y in ((1, 2, 3) if e else (0, 3)):
                    rec(t + 1, (0 if y == 1 else 1 if y == 0 else e), acc + [y])
            rec(0, e0[i], [])
            per_entity_opts.append(seqs)
        for cfg in itertools.product(*per_entity_opts):
            lab = torch.tensor([[cfg[i][t] for i in range(n)] for t in range(T)])
            s = float(crf_gold_score(phi, psi, init, lab, torch.tensor(e0)))
            scores.append(s)
            if s > best:
                best, best_cfg = s, (lab, e0)
    return torch.logsumexp(torch.tensor(scores, dtype=torch.float64), 0), best, best_cfg


# ============================================================================ đặc trưng văn bản
def word_tokens(sentence: str) -> List[str]:
    return sentence.split()


def mention_word_mask(entity: str, words: List[str], mode: str = "report") -> List[bool]:
    """Từ nào trong câu khớp từ nội dung của tên thực thể (cùng quy tắc cờ nhắc của core)."""
    out = []
    for w in words:
        out.append(pc.is_mentioned(entity, w, mode))
    return out


def build_entity_input(tok, rec: dict, i: int, max_len: int = 512):
    """<s> tên thực thể </s></s> câu1 câu2 ... </s>; trả về ids, span mỗi câu, chỉ số token được nhắc mỗi câu."""
    ent = rec["entities"][i]
    ids = [tok.cls_token_id] + tok.encode(" " + ent.replace(";", " /"), add_special_tokens=False)[:30] + [tok.sep_token_id, tok.sep_token_id]
    name_span = (1, len(ids) - 2)
    spans, ment, word_toks = [], [], []
    for t, s in enumerate(rec["sentences"]):
        start = len(ids)
        mt, wt = [], []
        for k, w in enumerate(word_tokens(s)):
            wid = tok.encode(" " + w, add_special_tokens=False)
            if pc.is_mentioned(ent, w):
                mt.extend(range(len(ids), len(ids) + len(wid)))
            wt.append((len(ids), len(ids) + len(wid)))
            ids.extend(wid)
        spans.append((start, len(ids)))
        ment.append(mt)
        word_toks.append(wt)
    ids.append(tok.sep_token_id)
    if len(ids) > max_len:
        raise ValueError(f"đoạn {rec['paragraph_id']} dài {len(ids)} > {max_len} token")
    return ids, spans, ment, name_span, word_toks


_NP_TAGS = {"NN", "NNS", "NNP", "NNPS", "JJ", "JJR", "JJS", "VBN", "VBG", "CD", "DT", "PRP$"}
_NOUN = {"NN", "NNS", "NNP", "NNPS"}
_LOC_STOP = {"the", "a", "an", "its", "their", "his", "her", "this", "that", "these", "those", "some", "any"}


def norm_loc(s: str) -> str:
    """Chuẩn hoá chuỗi vị trí để so khớp: chữ thường, bỏ mạo từ/sở hữu, đưa về số ít đơn giản."""
    ws = [w for w in re.findall(r"[a-z0-9]+", s.lower()) if w not in _LOC_STOP]
    return " ".join(w[:-1] if len(w) > 3 and w.endswith("s") and not w.endswith("ss") else w for w in ws)


def location_candidates(rec: dict, tagger=None) -> List[dict]:
    """Ứng viên vị trí (mục 4.3 báo cáo: 'đầu riêng chấm điểm trên các cụm danh từ').
    Mỗi cụm danh từ cho 3 ứng viên: cả cụm (bỏ mạo từ), 2 từ cuối, từ cuối. Gộp theo chuỗi chuẩn hoá.
    Không có tagger: lùi về cửa sổ 1–2 từ không phải từ rỗng."""
    cands: Dict[str, dict] = {}

    def add(norm, surface, occ):
        if not norm:
            return
        c = cands.setdefault(norm, {"norm": norm, "surface": surface, "occ": []})
        c["occ"].append(occ)

    for t, s in enumerate(rec["sentences"]):
        words = word_tokens(s)
        clean = [re.sub(r"[^A-Za-z0-9\-']", "", w) for w in words]
        if tagger is not None:
            idx = [k for k, w in enumerate(clean) if w]
            tags = dict(zip(idx, [tg for _, tg in tagger([clean[k] for k in idx])])) if idx else {}
            k = 0
            while k < len(words):
                if tags.get(k) in _NP_TAGS:
                    j = k + 1
                    while j < len(words) and tags.get(j) in _NP_TAGS and tags.get(j) not in ("DT", "PRP$"):
                        j += 1
                    last_noun = max((q for q in range(k, j) if tags.get(q) in _NOUN), default=None)
                    if last_noun is not None:
                        first = k
                        while first <= last_noun and clean[first].lower() in _LOC_STOP:
                            first += 1
                        for a in {first, max(first, last_noun - 1), last_noun}:
                            phrase = " ".join(clean[a:last_noun + 1]).lower()
                            add(norm_loc(phrase), phrase, (t, a, last_noun + 1))
                    k = j
                else:
                    k += 1
        else:
            for k in range(len(words)):
                for L in (1, 2):
                    seg = clean[k:k + L]
                    if len(seg) == L and all(w and w.lower() not in pc.STOPWORDS for w in seg):
                        phrase = " ".join(seg).lower()
                        add(norm_loc(phrase), phrase, (t, k, k + L))
    return list(cands.values())


def last_mention_distance(rec: dict, i: int, t: int) -> float:
    """d_it: số câu kể từ lần nhắc gần nhất của thực thể i trước câu t (chưa từng nhắc → T)."""
    T = len(rec["sentences"])
    for k in range(t - 1, 0, -1):
        if rec["mentions"][i][k - 1]:
            return float(t - k)
    return float(T)


# ============================================================================ mô hình
class PP4Model(nn.Module):
    """variant ∈ {"M0", "CRF_NOPAIR", "CRF_NOE", "CRF_FULL"}; M0 = phân loại từng ô độc lập (cross-entropy)."""

    def __init__(self, encoder, hidden: int, variant: str, lstm_hidden: int = 256, dropout: float = 0.1):
        super().__init__()
        self.encoder = encoder
        self.variant = variant
        self.lstm = nn.LSTM(hidden, lstm_hidden, batch_first=True, bidirectional=True)
        H = 2 * lstm_hidden
        self.att_q = nn.Linear(H, H)
        self.drop = nn.Dropout(dropout)
        self.phi = nn.Sequential(nn.Linear(H + 1, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 4))
        self.init_head = nn.Linear(H, 1)
        self.pair_w = nn.Linear(6, 1)        # [log E, m_i, m_j, d_i/T, cos, 1]
        nn.init.zeros_(self.pair_w.weight)
        nn.init.zeros_(self.pair_w.bias)
        self.loc_q = nn.Linear(H + 1, H)      # đầu vị trí: chấm điểm ứng viên cụm danh từ
        self.loc_unk = nn.Linear(H + 1, 1)    # lựa chọn '?' (tồn tại nhưng không rõ vị trí)
        self.lambda_loc = 1.0

    def entity_states(self, batch_ids, attn, spans, ment, name_spans):
        out = self.encoder(input_ids=batch_ids, attention_mask=attn).last_hidden_state
        out, _ = self.lstm(out.float())
        out = self.drop(out)
        n = out.shape[0]
        T = len(spans[0])
        H = out.shape[-1]
        h = out.new_zeros(T, n, H)
        m = out.new_zeros(T, n)
        pooled = out.new_zeros(n, H)
        for i in range(n):
            a, b = name_spans[i]
            q = self.att_q(out[i, a:b].mean(0))
            pooled[i] = out[i, spans[i][0][0]:spans[i][-1][1]].mean(0)
            for t, (s, e) in enumerate(spans[i]):
                if ment[i][t]:
                    h[t, i] = out[i, ment[i][t]].mean(0)
                    m[t, i] = 1.0
                else:
                    seg = out[i, s:e]
                    w = torch.softmax(seg @ q / math.sqrt(H), 0)
                    h[t, i] = (w[:, None] * seg).sum(0)
        return h, m, pooled, out

    def potentials(self, feats: dict, device):
        h, m, pooled, out = self.entity_states(feats["ids"].to(device), feats["attn"].to(device), feats["spans"],
                                               feats["ment"], feats["name_spans"])
        T, n, _ = h.shape
        phi = self.phi(torch.cat([h, m[..., None]], -1))                                 # (T,n,4)
        init = self.init_head(pooled).squeeze(-1)                                         # (n,)
        psi = None
        if self.variant in ("CRF_NOE", "CRF_FULL"):
            hn = F.normalize(h, dim=-1)
            cos = torch.einsum("tid,tjd->tij", hn, hn)
            logE = feats["logE"].to(device) if self.variant == "CRF_FULL" else torch.zeros(n, n, device=device)
            mi = m[:, :, None].expand(T, n, n)
            mj = m[:, None, :].expand(T, n, n)
            d = feats["dist"].to(device)[:, :, None].expand(T, n, n)
            f = torch.stack([logE[None].expand(T, n, n), mi, mj, d, cos, torch.ones_like(cos)], -1)
            psi = self.pair_w(f).squeeze(-1)
        loc = None
        if feats.get("cand_tok") and len(feats["cand_tok"][0]) > 0:
            hm = torch.cat([h, m[..., None]], -1)                                        # (T,n,H+1)
            q = self.loc_q(hm)                                                            # (T,n,H)
            K = len(feats["cand_tok"][0])
            loc = h.new_zeros(T, n, K + 1)
            for i in range(n):
                C = torch.stack([out[i, toks].mean(0) for toks in feats["cand_tok"][i]])  # (K,H)
                loc[:, i, :K] = q[:, i] @ C.T / math.sqrt(C.shape[-1])
            loc[:, :, K] = self.loc_unk(hm).squeeze(-1)
        self._loc = loc
        return phi, psi, init

    def loss(self, feats: dict, device):
        phi, psi, init = self.potentials(feats, device)
        gl, gi = feats["gold_lab"].to(device), feats["gold_init"].to(device)
        if self.variant == "M0":
            ce = F.cross_entropy(phi.reshape(-1, 4), gl.reshape(-1), reduction="sum")
            bce = F.binary_cross_entropy_with_logits(init, gi.float(), reduction="sum")
            main = (ce + bce) / gl.numel()
        else:
            main = crf_nll(phi.float(), None if psi is None else psi.float(), init.float(), gl, gi) / gl.numel()
        # L_total = L_PP4 + λ_loc · L_vị trí  (chỉ trên ô vàng tồn tại sau câu và có vị trí khớp ứng viên)
        if self._loc is not None:
            tgt = feats["loc_target"].to(device)
            ok = tgt >= 0
            if ok.any():
                main = main + self.lambda_loc * F.cross_entropy(self._loc.float()[ok], tgt[ok], reduction="sum") / gl.numel()
        return main

    @torch.no_grad()
    def predict(self, feats: dict, device, constrained: Optional[bool] = None):
        """M0: argmax từng ô (constrained=False) hoặc Viterbi với log-softmax (M0c, constrained=True).
        CRF: luôn Viterbi."""
        phi, psi, init = self.potentials(feats, device)
        if self.variant == "M0" and not constrained:
            lab = phi.argmax(-1).T.tolist()
            labels = [[IDX_LAB[y] for y in row] for row in lab]
            init_e = (init > 0).tolist()
        else:
            if self.variant == "M0":
                phi = F.log_softmax(phi.float(), -1)
                init = F.logsigmoid(init.float()) - F.logsigmoid(-init.float())
            labels, init_e, _ = crf_viterbi(phi.float(), None if psi is None else psi.float(), init.float())
        la = self._decode_locations(labels, init_e, feats)
        return labels, init_e, (la, self._init_loc)

    def _decode_locations(self, labels, init_e, feats):
        """Vị trí sau mỗi câu, nhất quán với nhãn: CREATE/MOVE → ứng viên điểm cao nhất; NONE → giữ; DESTROY → '-'."""
        n, T = len(labels), len(labels[0])
        surf = [c for c in feats.get("cand_surface", [])]
        best = self._loc.argmax(-1).tolist() if self._loc is not None else None
        out, self._init_loc = [], []
        for i in range(n):
            cur = pc.NO_LOC
            if init_e[i]:                     # vị trí ban đầu = ứng viên điểm cao nhất ở câu 1
                k0 = best[0][i] if best is not None else len(surf)
                cur = surf[k0] if k0 < len(surf) else "?"
            self._init_loc.append(cur)
            row = []
            for t in range(T):
                y = labels[i][t]
                if y == pc.DESTROY:
                    cur = pc.NO_LOC
                elif y in (pc.CREATE, pc.MOVE):
                    k = best[t][i] if best is not None else len(surf)
                    cur = surf[k] if k < len(surf) else "?"
                row.append(cur)
            out.append(row)
        return out

    @torch.no_grad()
    def existence_probs(self, feats: dict, device) -> torch.Tensor:
        """p(tồn tại) trước/sau mỗi câu (T+1, n) từ biên CRF; M0 dùng log-softmax trong cùng khung CRF (PP2, công thức 11)."""
        phi, psi, init = self.potentials(feats, device)
        if self.variant == "M0":
            phi = F.log_softmax(phi.float(), -1)
            init = F.logsigmoid(init.float()) - F.logsigmoid(-init.float())
        return crf_existence_marginals(phi.float(), None if psi is None else psi.float(), init.float())


def featurize(tok, rec: dict, logE: Optional[List[List[float]]] = None, max_len: int = 512, tagger=None) -> dict:
    n, T = len(rec["entities"]), len(rec["sentences"])
    seqs, spans, ment, names, wtoks = [], [], [], [], []
    for i in range(n):
        ids, sp, mt, ns, wt = build_entity_input(tok, rec, i, max_len)
        seqs.append(ids); spans.append(sp); ment.append(mt); names.append(ns); wtoks.append(wt)
    cands = location_candidates(rec, tagger)
    cand_tok = [[sorted({x for (t, a, b) in c["occ"] for w in range(a, b) for x in range(*wtoks[i][t][w])}) for c in cands]
                for i in range(n)]
    norm2k = {c["norm"]: k for k, c in enumerate(cands)}
    K = len(cands)
    loc_target = torch.full((T, n), -1, dtype=torch.long)
    n_loc, n_hit = 0, 0
    for i in range(n):
        for t in range(1, T + 1):
            st = rec["states"][i][t]
            if st == pc.NO_LOC:
                continue
            n_loc += 1
            if st == pc.UNK_LOC:
                loc_target[t - 1, i] = K; n_hit += 1
            elif norm_loc(st) in norm2k:
                loc_target[t - 1, i] = norm2k[norm_loc(st)]; n_hit += 1
    L = max(len(s) for s in seqs)
    ids = torch.full((n, L), tok.pad_token_id, dtype=torch.long)
    attn = torch.zeros((n, L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s)
        attn[i, :len(s)] = 1
    gold_lab = torch.tensor([[LAB_IDX[rec["labels"][i][t]] for i in range(n)] for t in range(T)])
    gold_init = torch.tensor([int(rec["existence"][i][0]) for i in range(n)])
    dist = torch.tensor([[last_mention_distance(rec, i, t) / T for i in range(n)] for t in range(1, T + 1)])
    E = torch.tensor(logE, dtype=torch.float) if logE is not None else torch.zeros(n, n)
    return {"ids": ids, "attn": attn, "spans": spans, "ment": ment, "name_spans": names,
            "gold_lab": gold_lab, "gold_init": gold_init, "dist": dist, "logE": E, "pid": rec["paragraph_id"],
            "cand_tok": cand_tok if K else [], "cand_surface": [c["surface"] for c in cands], "loc_target": loc_target,
            "loc_cover": (n_hit, n_loc)}


def to_prediction(rec: dict, labels, init_exists, locs=None) -> dict:
    p = {"paragraph_id": rec["paragraph_id"], "labels": labels, "exists_initial": [bool(x) for x in init_exists]}
    if locs is not None:
        p["locations_after"], p["location_initial"] = locs
    return p


# ============================================================================ huấn luyện
def train_variant(make_encoder, tok, variant: str, train_feats: List[dict], dev_recs: List[dict], dev_feats: List[dict],
                  evaluate_fn, seed: int, epochs: int = 8, lr_enc: float = 2e-5, lr_head: float = 1e-3,
                  accum: int = 4, device: str = "cuda", log=print, use_amp: bool = True,
                  ckpt_path: Optional[str] = None, patience: Optional[int] = None, stop_after_epochs: Optional[int] = None):
    """Huấn luyện một biến thể; chọn epoch theo PAIR_F1 trên DEV (quy tắc khoá trước; hoà → dev loss thấp hơn).
    ckpt_path: sau MỖI epoch lưu đủ trạng thái (mô hình, optimizer, scheduler, scaler, RNG, lịch sử, bản tốt nhất)
               → chạy lại sẽ TIẾP TỤC từ epoch kế tiếp. Người gọi xoá file khi đã lưu kết quả.
    patience:  dừng sớm khi PAIR_F1 dev (tốt nhất trong các tên theo dõi) không tăng sau `patience` epoch liên tiếp.
    stop_after_epochs: chỉ dùng để KIỂM THỬ việc bị ngắt (giả lập phiên chết sau N epoch của lần gọi này).
    Với M0: theo dõi ĐỒNG THỜI M0 (argmax) và M0c (Viterbi ràng buộc trên cùng mô hình), mỗi cái chọn epoch riêng.
    Trả về (model, lịch sử, {tên: (state_dict tốt nhất, epoch tốt nhất)})."""
    random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    enc, hidden = make_encoder()
    model = PP4Model(enc, hidden, variant).to(device)
    enc_ids = {id(p) for p in model.encoder.parameters()}
    head_params = [p for p in model.parameters() if id(p) not in enc_ids]
    opt = torch.optim.AdamW([{"params": list(model.encoder.parameters()), "lr": lr_enc}, {"params": head_params, "lr": lr_head}],
                            weight_decay=0.01)
    steps = max(1, epochs * math.ceil(len(train_feats) / accum))
    warm = max(1, int(0.1 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda st: min(1.0, (st + 1) / warm) * max(0.0, (steps - st) / steps))
    amp = use_amp and device.startswith("cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    names = ["M0", "M0c"] if variant == "M0" else [variant]
    best = {nm: ((-1.0, -float("inf")), None, None) for nm in names}
    hist, order = [], list(range(len(train_feats)))
    start_ep, bad, best_main = 1, 0, -1.0
    if ckpt_path and os.path.exists(ckpt_path):
        ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
        scaler.load_state_dict(ck["scaler"]); hist, order = ck["hist"], ck["order"]
        for nm in names:                      # bản tốt nhất được lưu riêng (chỉ ghi khi cải thiện)
            key_, ep_ = ck["best_meta"][nm]
            bp = f"{ckpt_path}.best_{nm}.pt"
            best[nm] = (tuple(key_), torch.load(bp, map_location="cpu", weights_only=True) if os.path.exists(bp) else None, ep_)
        bad, best_main, start_ep = ck["bad"], ck["best_main"], ck["epoch"] + 1
        random.setstate(ck["py_rng"]); torch.set_rng_state(ck["torch_rng"])
        if torch.cuda.is_available() and ck.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all(ck["cuda_rng"])
        log(f"  [{variant} seed={seed}] TIẾP TỤC từ checkpoint epoch {ck['epoch']}" + (" (đã dừng sớm)" if ck.get("stopped") else ""))
        if ck.get("stopped"):
            start_ep = epochs + 1
    n_run = 0
    for ep in range(start_ep, epochs + 1):
        model.train()
        random.shuffle(order)
        t0, tot = time.time(), 0.0
        opt.zero_grad()
        for k, j in enumerate(order, 1):
            with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                loss = model.loss(train_feats[j], device)
            scaler.scale(loss / accum).backward()
            tot += float(loss.detach())
            if k % accum == 0 or k == len(order):
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        model.eval()
        dev_loss, preds = 0.0, {nm: {} for nm in names}
        with torch.no_grad():
            for rec, f in zip(dev_recs, dev_feats):
                with torch.autocast("cuda", dtype=torch.float16, enabled=amp):
                    dev_loss += float(model.loss(f, device))
                for nm in names:
                    lab, ie, la = model.predict(f, device, constrained=(nm == "M0c"))
                    preds[nm][rec["paragraph_id"]] = to_prediction(rec, lab, ie, la)
        row = {"epoch": ep, "train_loss": tot / len(order), "dev_loss": dev_loss / len(dev_recs), "sec": round(time.time() - t0, 1)}
        for nm in names:
            res = evaluate_fn(preds[nm], dev_recs)
            row[f"{nm}:dev_PAIR_F1"] = res["PAIR@all"]["F1"]
            row[f"{nm}:dev_PAIR_F1@any_imp"] = res["PAIR@any_implicit"]["F1"]
            row[f"{nm}:dev_IMP_ACC_DESTROY"] = res["IMPLICIT_ACC"]["DESTROY"]["acc"]
            row[f"{nm}:dev_IMP_ACC_CREATE"] = res["IMPLICIT_ACC"]["CREATE"]["acc"]
            row[f"{nm}:dev_PAIR_LOC_F1"] = res["PAIR_LOC"]["F1"]
            key = (res["PAIR@all"]["F1"], -row["dev_loss"])
            if key > best[nm][0]:
                best[nm] = (key, {kk: v.detach().to("cpu", copy=True) for kk, v in model.state_dict().items()}, ep)
                if ckpt_path:                    # ghi bản tốt nhất CHỈ khi cải thiện (tránh ghi 2,5 GB mỗi epoch)
                    torch.save(best[nm][1], f"{ckpt_path}.best_{nm}.pt.tmp")
                    os.replace(f"{ckpt_path}.best_{nm}.pt.tmp", f"{ckpt_path}.best_{nm}.pt")
        hist.append(row)
        cur = max(row[f"{nm}:dev_PAIR_F1"] for nm in names)
        if cur > best_main + 1e-9:
            best_main, bad = cur, 0
        else:
            bad += 1
        stopped = patience is not None and bad >= patience and ep < epochs
        log(f"  [{variant} seed={seed}] ep{ep} train={row['train_loss']:.4f} dev={row['dev_loss']:.4f} " +
            " ".join(f"{nm}:F1={row[nm + ':dev_PAIR_F1']:.3f}/imp={row[nm + ':dev_PAIR_F1@any_imp']:.3f}" for nm in names) +
            f" ({row['sec']}s)" + (f" → DỪNG SỚM (không tăng {bad} epoch)" if stopped else ""))
        if ckpt_path:
            tmp = ckpt_path + ".tmp"
            torch.save({"epoch": ep, "model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                        "scaler": scaler.state_dict(), "hist": hist, "order": order,
                        "best_meta": {nm: (list(b[0]), b[2]) for nm, b in best.items()}, "bad": bad,
                        "best_main": best_main, "stopped": stopped, "py_rng": random.getstate(), "torch_rng": torch.get_rng_state(),
                        "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None}, tmp)
            os.replace(tmp, ckpt_path)
        n_run += 1
        if stopped:
            break
        if stop_after_epochs is not None and n_run >= stop_after_epochs and ep < epochs:
            raise KeyboardInterrupt(f"giả lập phiên bị ngắt sau epoch {ep}")
    return model, hist, {nm: (b[1], b[2]) for nm, b in best.items()}


def cleanup_ckpt(ckpt_path: str) -> None:
    """Xoá checkpoint đang-chạy và các file bản-tốt-nhất đi kèm (gọi sau khi đã lưu kết quả)."""
    import glob
    for f in [ckpt_path] + glob.glob(ckpt_path + ".best_*.pt*"):
        if os.path.exists(f):
            os.remove(f)


@torch.no_grad()
def predict_all(model, recs: List[dict], feats: List[dict], device: str, constrained: Optional[bool] = None) -> Dict[str, dict]:
    model.eval()
    out = {}
    for rec, f in zip(recs, feats):
        lab, ie, la = model.predict(f, device, constrained=constrained)
        out[rec["paragraph_id"]] = to_prediction(rec, lab, ie, la)
    return out
'''
MODULES['propara_pp4_tests.py'] = r'''
"""Kiểm thử CRF của PP4 bằng liệt kê vét cạn (cần torch, không cần GPU/transformers)."""
import random

import torch

import propara_core as pc
import propara_pp4 as pp


def _rand(n, T, seed, with_psi=True):
    g = torch.Generator().manual_seed(seed)
    phi = torch.randn(T, n, 4, generator=g, dtype=torch.float64)
    psi = torch.randn(T, n, n, generator=g, dtype=torch.float64) if with_psi else None
    init = torch.randn(n, generator=g, dtype=torch.float64)
    return phi, psi, init


def test_log_partition_matches_brute_force():
    for (n, T, seed) in [(1, 3, 0), (2, 2, 1), (2, 3, 2), (3, 2, 3)]:
        for with_psi in (False, True):
            phi, psi, init = _rand(n, T, seed, with_psi)
            z = pp.crf_log_partition(phi, psi, init)
            zb, _, _ = pp.brute_force(phi, psi, init)
            assert abs(float(z) - float(zb)) < 1e-8, (n, T, with_psi, float(z), float(zb))


def test_viterbi_matches_brute_force():
    for (n, T, seed) in [(2, 2, 4), (2, 3, 5), (3, 2, 6)]:
        phi, psi, init = _rand(n, T, seed)
        labels, init_e, best = pp.crf_viterbi(phi, psi, init)
        _, bb, (lab, e0) = pp.brute_force(phi, psi, init)
        assert abs(best - bb) < 1e-8
        assert [[pp.IDX_LAB[int(lab[t, i])] for t in range(T)] for i in range(n)] == labels
        assert init_e == [bool(x) for x in e0]


def test_viterbi_output_always_satisfies_constraints():
    for seed in range(20):
        n, T = random.Random(seed).randint(1, 4), random.Random(seed + 1).randint(1, 5)
        phi, psi, init = _rand(n, T, 100 + seed)
        labels, init_e, _ = pp.crf_viterbi(phi * 5, psi, init)
        assert pc.count_constraint_violations(labels, init_e)[0] == 0


def test_nll_nonnegative_and_gradient_flows():
    phi, psi, init = _rand(3, 3, 7)
    phi.requires_grad_(True); psi.requires_grad_(True)
    gl = torch.tensor([[3, 3, 0], [1, 3, 3], [3, 3, 3]])          # e0=(1,1,0): t1 C cho thực thể 2, t2 D cho thực thể 0
    gi = torch.tensor([1, 1, 0])
    nll = pp.crf_nll(phi, psi, init, gl, gi)
    assert float(nll.detach()) >= 0
    nll.backward()
    assert phi.grad is not None and float(psi.grad.abs().sum()) > 0


def test_pair_score_rewards_simultaneous_destroy_create():
    # φ trung tính; ψ(0→1) lớn ở câu 1 → Viterbi chọn 0 bị DESTROY và 1 được CREATE cùng lúc
    phi = torch.zeros(1, 2, 4, dtype=torch.float64)
    psi = torch.zeros(1, 2, 2, dtype=torch.float64); psi[0, 0, 1] = 5.0
    init = torch.tensor([3.0, -3.0], dtype=torch.float64)
    labels, init_e, _ = pp.crf_viterbi(phi, psi, init)
    assert init_e == [True, False] and labels[0][0] == pc.DESTROY and labels[1][0] == pc.CREATE


def test_existence_marginals_match_brute_force():
    import itertools
    for (n, T, seed) in [(2, 2, 11), (3, 2, 12), (2, 3, 13)]:
        phi, psi, init = _rand(n, T, seed)
        m = pp.crf_existence_marginals(phi, psi, init)
        # vét cạn: liệt kê mọi cấu hình hợp lệ, cộng trọng số theo tồn tại
        acc = torch.zeros(T + 1, n, dtype=torch.float64); Z = 0.0
        for e0 in itertools.product([0, 1], repeat=n):
            opts = []
            for i in range(n):
                seqs = []
                def rec(tt, e, a):
                    if tt == T:
                        seqs.append(tuple(a)); return
                    for y in ((1, 2, 3) if e else (0, 3)):
                        rec(tt + 1, (0 if y == 1 else 1 if y == 0 else e), a + [y])
                rec(0, e0[i], [])
                opts.append(seqs)
            for cfg in itertools.product(*opts):
                lab = torch.tensor([[cfg[i][tt] for i in range(n)] for tt in range(T)])
                w = float(torch.exp(pp.crf_gold_score(phi, psi, init, lab, torch.tensor(e0))))
                Z += w
                for i in range(n):
                    e = e0[i]; acc[0, i] += w * e
                    for tt in range(T):
                        y = cfg[i][tt]; e = 0 if y == 1 else 1 if y == 0 else e
                        acc[tt + 1, i] += w * e
        assert torch.allclose(m, acc / Z, atol=1e-8), (m, acc / Z)


def run_all(verbose=True):
    names = sorted(n for n in globals() if n.startswith("test_"))
    failed = []
    for n in names:
        try:
            globals()[n]()
            if verbose:
                print(f"  PASS  {n}")
        except Exception as e:  # noqa: BLE001
            failed.append((n, repr(e)))
            print(f"  FAIL  {n}: {e!r}")
    return len(names), failed
'''
MODULES['propara_robust.py'] = r'''
"""propara_robust — biến thể kiểm tra khái quát (Tập D trong PIPELINE §10.2).

Phiên bản: robust_v1.
NONCE: thay mọi TỪ NỘI DUNG trong tên thực thể bằng từ vô nghĩa, nhất quán trong toàn đoạn (câu, tên thực thể,
vị trí). Nhãn không đổi (đã kiểm tra lại sau khi thay). Đây là phép thử MẤT TRI THỨC NGỮ NGHĨA: điểm giảm KHÔNG
một mình chứng minh mô hình 'chỉ ghi nhớ' (xem PIPELINE §10.2, sai lầm #10).
"""
from __future__ import annotations

import re
from typing import Dict, List, Optional, Tuple

import propara_core as pc

ROBUST_VERSION = "robust_v1"
_SYL_A = ["bl", "dr", "fr", "gl", "kr", "pl", "sn", "tr", "v", "z", "m", "n", "t", "d", "k", "p"]
_SYL_B = ["a", "e", "i", "o", "u", "ai", "ou"]
_SYL_C = ["x", "rb", "nd", "mp", "sk", "lt", "rk", "v", "z", "t", "p", "g"]


def nonce_word(k: int) -> str:
    a = _SYL_A[k % len(_SYL_A)]
    b = _SYL_B[(k // len(_SYL_A)) % len(_SYL_B)]
    c = _SYL_C[(k // (len(_SYL_A) * len(_SYL_B))) % len(_SYL_C)]
    return a + b + c + ("o" if k >= len(_SYL_A) * len(_SYL_B) * len(_SYL_C) else "")


def _content_words(entity: str) -> List[str]:
    return [w for a in pc.split_aliases(entity) for w in pc.tokenize(a) if w not in pc.STOPWORDS and not w.isdigit()]


def nonce_variant(raw: dict, vocab: Optional[set] = None, offset: int = 0) -> Tuple[Optional[dict], Dict[str, str]]:
    """Trả về (raw mới, ánh xạ) hoặc (None, ...) nếu nhãn bị thay đổi (khi đó bỏ đoạn khỏi biến thể)."""
    words = []
    for e in raw["participants"]:
        for w in _content_words(e):
            base = w[:-1] if w.endswith("s") and len(w) > 3 else w
            if base not in words:
                words.append(base)
    vocab = vocab or set()
    mapping, k = {}, offset
    for w in words:
        nw = nonce_word(k)
        while nw in vocab or nw in mapping.values():
            k += 1
            nw = nonce_word(k)
        mapping[w] = nw
        k += 1

    def sub(text: str) -> str:
        def rep(mt):
            tokw = mt.group(0)
            low = tokw.lower()
            stem, plural = (low[:-1], True) if low.endswith("s") and low[:-1] in mapping else (low, False)
            if stem not in mapping:
                return tokw
            out = mapping[stem] + ("s" if plural else "")
            return out.capitalize() if tokw[0].isupper() else out
        return re.sub(r"[A-Za-z]+", rep, text)

    new = {"para_id": raw["para_id"], "sentence_texts": [sub(s) for s in raw["sentence_texts"]],
           "participants": [sub(p) for p in raw["participants"]],
           "states": [[v if v in (pc.NO_LOC, pc.UNK_LOC) else sub(v) for v in st] for st in raw["states"]]}
    old_l = [[pc.derive_label(st[t - 1], st[t]) for t in range(1, len(st))] for st in raw["states"]]
    new_l = [[pc.derive_label(st[t - 1], st[t]) for t in range(1, len(st))] for st in new["states"]]
    if old_l != new_l:
        return None, mapping
    return new, mapping


def nonce_records(raws_by_split: Dict[str, List[dict]], topics: Dict[str, str]) -> Dict[str, List[dict]]:
    vocab = set()
    for rows in raws_by_split.values():
        for r in rows:
            for s in r["sentence_texts"]:
                vocab.update(pc.tokenize(s))
    out = {}
    for sp, rows in raws_by_split.items():
        recs = []
        for r in rows:
            nr, _ = nonce_variant(r, vocab)
            if nr is not None:
                rec = pc.build_record(nr, sp, topics.get(str(r["para_id"])))
                rec["variant"] = "nonce_v1"
                recs.append(rec)
        out[sp] = recs
    return out
'''
BLOBS = {}
BLOBS['aristo_propara_evaluator.zip'] = ('...
AAC9AgIAAAA=
''')

import base64 as _b64, zipfile as _zf, io as _io, importlib
MOD_DIR = OUT / "code"
MOD_DIR.mkdir(parents=True, exist_ok=True)
for _name, _src in MODULES.items():
    (MOD_DIR / _name).write_text(_src, encoding="utf-8")
for _name, (_sha, _b) in BLOBS.items():
    _data = _b64.b64decode("".join(_b.split()))
    if hashlib.sha256(_data).hexdigest() != _sha:
        raise RuntimeError(f"Blob {_name} hỏng (sha256 lệch)")
    (MOD_DIR / _name).write_bytes(_data)
    if _name.endswith(".zip"):
        with _zf.ZipFile(_io.BytesIO(_data)) as z:
            z.extractall(MOD_DIR)
sys.dont_write_bytecode = True
if str(MOD_DIR) not in sys.path:
    sys.path.insert(0, str(MOD_DIR))
for _m in [n[:-3] for n in MODULES]:
    if _m in sys.modules:
        importlib.reload(sys.modules[_m])
print("Đã nạp:", ", ".join(MODULES), "|", ", ".join(BLOBS))

import propara_core as pc, propara_metrics as pm, propara_heuristics as ph, propara_pp3 as p3, propara_pp4 as pp, propara_robust as pr
import propara_core_tests as pct, propara_metrics_tests as pmt, propara_pp4_tests as ppt
EVAL_DIR = str(MOD_DIR / "aristo_propara_evaluator")
_f = []
for _nm, _m in (("core", pct), ("metrics", pmt), ("pp4/CRF", ppt)):
    _n, _ff = _m.run_all(verbose=False); print(f"{_nm}: {_n - len(_ff)}/{_n}"); _f += _ff
if _f:
    raise RuntimeError(f"Kiểm thử thất bại: {_f}")

## 4. Dữ liệu

In [ ]:
def find_in_input(suffixes):
    if not INPUT_ROOT.exists():
        return None
    for suf in suffixes:
        for p in sorted(INPUT_ROOT.rglob(suf.split("/")[-1])):
            if p.is_file() and p.as_posix().endswith(suf):
                return p
    return None

RAW, RAW_SOURCE, CHECKSUM = {}, {}, {}
(OUT / "raw").mkdir(parents=True, exist_ok=True)
for name, (url, sha, suffixes) in RAW_FILES.items():
    p = find_in_input(suffixes)
    if p is not None:
        RAW_SOURCE[name] = f"input:{p}"
    else:
        p = OUT / "raw" / name
        if not p.exists():
            if ENV["internet"] is not True:
                raise FileNotFoundError(f"Không tìm thấy {name} trong {INPUT_ROOT} và không có Internet. "
                                        "Bật Settings → Internet, hoặc Add Input dataset propara-raw-v1.")
            for attempt in range(3):
                try:
                    urllib.request.urlretrieve(url, p)
                    break
                except Exception as e:
                    if attempt == 2:
                        raise RuntimeError(f"Tải thất bại {url}: {e}")
                    time.sleep(2)
        RAW_SOURCE[name] = f"download:{url}"
    RAW[name] = p
    got = pc.sha256_file(str(p))
    CHECKSUM[name] = {"expected": sha, "observed": got, "match": got == sha}
    print(f"{'OK ' if got == sha else 'LỆCH'}  {name}")

TOPICS = pc.load_topics_from_naacl_grid(str(RAW["gold-full-grids.v3.tsv"]))
RECS = [pc.build_record(r, sp, TOPICS.get(str(r["para_id"])))
        for sp in SPLITS for r in pc.load_jsonl(str(RAW[f"grids.v1.{sp}.json"]))]
BY_SPLIT = {sp: [r for r in RECS if r["split"] == sp] for sp in SPLITS}
BY_PID = {r["paragraph_id"]: r for r in RECS}
_n = (len(RECS), sum(len(r["sentences"]) for r in RECS), sum(len(r["entities"]) for r in RECS))
_pairs = {sp: sum(len(pc.gold_pairs_report(r)) for r in BY_SPLIT[sp]) for sp in SPLITS}
print("đoạn/câu/thực thể:", _n, "| cặp vàng:", _pairs)
if _n != (488, 3302, 1915) or _pairs != {"train": 378, "dev": 58, "test": 97}:
    raise RuntimeError("Dữ liệu dựng lại KHÔNG khớp kết quả notebook 00 (488/3302/1915; 378/58/97). Dừng.")

TRAIN = BY_SPLIT["train"][:LIMIT_TRAIN] if LIMIT_TRAIN else BY_SPLIT["train"]
DEV = BY_SPLIT["dev"][:max(5, LIMIT_TRAIN // 2)] if LIMIT_TRAIN else BY_SPLIT["dev"]
TEST = BY_SPLIT["test"][:max(5, LIMIT_TRAIN // 2)] if LIMIT_TRAIN else BY_SPLIT["test"]
FOLDS = pc.cv_folds_by_topic(RECS, k=5, seed=SEED)
print("train/dev/test:", len(TRAIN), len(DEV), len(TEST))
STATE = OUT / "state"; STATE.mkdir(exist_ok=True)
def tdone(): return (time.time() - T0) / 3600

# TIẾP TỤC SANG PHIÊN MỚI: nếu đã Add Input output của một phiên bản trước của notebook này,
# chép các phần đã xong (kết quả, dự đoán, test kín, checkpoint) vào OUT rồi bỏ qua chúng khi chạy.
RESTORED = []
if INPUT_ROOT.exists():
    for prev in sorted(INPUT_ROOT.rglob(f"{RUN_TAG}/state/results.json")):
        prev_out = prev.parent.parent
        for sub in ("state", "predictions", "sealed_test_predictions", "checkpoints"):
            if (prev_out / sub).exists():
                for f in (prev_out / sub).rglob("*"):
                    if f.is_file():
                        dst = OUT / sub / f.relative_to(prev_out / sub)
                        if not dst.exists():
                            dst.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dst); RESTORED.append(str(dst.relative_to(OUT)))
        print(f"Khôi phục từ phiên trước: {prev_out} → {len(RESTORED)} file")
        break

## 5. Encoder `roberta-base` (ghim revision) + kiểm tra độ dài đầu vào

In [ ]:
from transformers import AutoTokenizer, AutoModel
from huggingface_hub import snapshot_download
_local_enc = os.environ.get("PROPARA_ENCODER_LOCAL", "")
ENC_PATH = _local_enc or snapshot_download(ENCODER_ID, revision=ENCODER_REV, allow_patterns=["*.json", "*.txt", "*.safetensors"])
tok = AutoTokenizer.from_pretrained(ENC_PATH)
def make_encoder():
    enc = AutoModel.from_pretrained(ENC_PATH)
    return enc, enc.config.hidden_size
_lens = [len(pp.build_entity_input(tok, r, i)[0]) for r in RECS for i in range(len(r["entities"]))]
print(f"độ dài đầu vào (thực thể + đoạn): TB {statistics.mean(_lens):.0f}, max {max(_lens)} token")
_emb_model = AutoModel.from_pretrained(ENC_PATH)
EMB_MATRIX = _emb_model.get_input_embeddings().weight.detach().float().cpu()
del _emb_model
def embed_words(words):
    """Embedding TĨNH (ma trận đầu vào roberta, trung bình sub-token) cho danh từ đầu, dùng cho E(X,Y)."""
    out = []
    for w in words:
        ids = tok.encode(" " + w, add_special_tokens=False) or [tok.unk_token_id]
        out.append(EMB_MATRIX[ids].mean(0))
    return torch.stack(out)

## 6. PP3: khai thác WikiHow → K, độ phủ trên 533 cặp vàng, huấn luyện E(X,Y)

In [ ]:
PP3 = {"status": "not_run"}
LOGE = {}
TAGGER = None
try:
    if WIKIHOW_LOCAL:
        wh_path = Path(WIKIHOW_LOCAL); PP3["source"] = f"local:{wh_path}"
    else:
        wh_path = Path.home() / "wikihowAll.csv"
        if not wh_path.exists():
            print("Đang tải WikiHow (~620 MB)…"); urllib.request.urlretrieve(WIKIHOW_URL, wh_path)
        _sha = pc.sha256_file(str(wh_path))
        PP3["source"] = WIKIHOW_URL; PP3["sha256_match"] = _sha == WIKIHOW_SHA
        print("SHA-256 khớp:", PP3["sha256_match"])
    # bộ gán nhãn từ loại (pp3_v2, cũng dùng để tách cụm danh từ cho đầu vị trí); thiếu thì lùi về regex và ghi cảnh báo
    try:
        try:
            import nltk
        except ImportError:                      # Kaggle thường có sẵn; nếu thiếu thì cài (cần Internet)
            import subprocess
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "nltk"], check=True)
            import nltk
        from nltk.tag import PerceptronTagger
        for _res in ("averaged_perceptron_tagger_eng", "averaged_perceptron_tagger"):
            try:
                TAGGER = PerceptronTagger().tag; break
            except LookupError:
                nltk.download(_res, quiet=True)
                try:
                    TAGGER = PerceptronTagger().tag; break
                except LookupError:
                    continue
    except Exception as _e:
        print("[CẢNH BÁO] không có NLTK tagger:", _e)
    PP3["extractor"] = "pos_nltk_perceptron (pp3_v2)" if TAGGER else "regex_only (pp3_v1, chất lượng thấp)"
    print("Bộ trích:", PP3["extractor"])
    csv.field_size_limit(10**9)
    def _rows():
        with open(wh_path, encoding="utf-8", errors="replace", newline="") as f:
            for row in csv.DictReader(f):
                yield row.get("title") or "", row.get("text") or ""
    t0 = time.time()
    PAIR_CNT, PAIR_ART, PAT_CNT, CORPUS, ART_NOUNS = p3.mine_corpus(_rows(), tagger=TAGGER)
    PP3.update({"mine_sec": round(time.time() - t0, 1), "corpus": CORPUS, "n_pairs_distinct": len(PAIR_CNT),
                "n_pair_mentions": sum(PAIR_CNT.values()), "pattern_counts": dict(PAT_CNT),
                "top_pairs": [f"{x}->{y}:{c}" for (x, y), c in PAIR_CNT.most_common(40)]})
    print({k: v for k, v in PP3.items() if k != "top_pairs"}); print("top:", PP3["top_pairs"][:20])
    GOLD_ALL = [g for r in RECS for g in pc.gold_pairs_report(r)]
    PP3["coverage_all_min1"] = p3.coverage(GOLD_ALL, set(PAIR_CNT))
    PP3["coverage_all_min2"] = p3.coverage(GOLD_ALL, {p for p, c in PAIR_CNT.items() if c >= 2})
    print("Độ phủ (24)–(26) trên 533 cặp vàng:", PP3["coverage_all_min1"], "| K với tần suất ≥2:", PP3["coverage_all_min2"])
    DATA, K = p3.build_training_pairs(PAIR_CNT, PAIR_ART, ART_NOUNS, seed=SEED)
    PP3["train_val_sizes"] = {k: len(v) for k, v in DATA.items()}
    print("Tập huấn luyện E:", PP3["train_val_sizes"])
    E_MODEL, E_HIST = p3.train_pair_scorer(DATA, embed_words, device=DEVICE, seed=SEED)
    E_MODEL = E_MODEL.to(DEVICE).eval()
    PP3["history"] = E_HIST
    PP3["best_val"] = min(E_HIST, key=lambda r: r["val_loss"])
    for r in RECS:
        LOGE[r["paragraph_id"]] = p3.logE_matrix(E_MODEL, embed_words, r, device=DEVICE)
    # E có phân biệt cặp vàng ProPara với cặp có thứ tự khác trong cùng đoạn không? (chẩn đoán, theo tập)
    for sp in SPLITS:
        posv, negv = [], []
        for r in BY_SPLIT[sp]:
            G = {(g["source_idx"], g["target_idx"]) for g in pc.gold_pairs_report(r)}
            n = len(r["entities"])
            for i in range(n):
                for j in range(n):
                    if i != j:
                        (posv if (i, j) in G else negv).append(LOGE[r["paragraph_id"]][i][j])
        PP3[f"propara_auc_{sp}"] = p3._auc(posv, negv)
    print("AUC của log E trên cặp ProPara (vàng vs cặp khác cùng đoạn):", {sp: round(PP3[f"propara_auc_{sp}"], 3) for sp in SPLITS})
    PP3["status"] = "ok"
except Exception as e:
    import traceback; traceback.print_exc()
    PP3["status"] = f"FAILED: {type(e).__name__}: {e}"
    print("[CẢNH BÁO] PP3 lỗi → bỏ CRF_FULL/CRF_RANDE, vẫn chạy các biến thể còn lại.")
if PP3["status"] != "ok":
    VARIANTS = [v for v in VARIANTS if v not in ("CRF_FULL", "CRF_RANDE")]
    CV_VARIANTS = [v if v != "CRF_FULL" else "CRF_NOE" for v in CV_VARIANTS]
    LOGE = {r["paragraph_id"]: [[0.0] * len(r["entities"]) for _ in r["entities"]] for r in RECS}
# M2r: xáo trộn log E giữa mọi cặp (giữ phân phối, phá tri thức), seed cố định
_vals = [v for r in RECS for i, row in enumerate(LOGE[r["paragraph_id"]]) for j, v in enumerate(row) if i != j]
random.Random(SEED).shuffle(_vals)
LOGE_RAND, _k = {}, 0
for r in RECS:
    n = len(r["entities"]); M = [[0.0] * n for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if i != j:
                M[i][j] = _vals[_k]; _k += 1
    LOGE_RAND[r["paragraph_id"]] = M

## 7. Đặc trưng cho PP4

In [ ]:
t0 = time.time()
FEATS = {r["paragraph_id"]: pp.featurize(tok, r, LOGE[r["paragraph_id"]], tagger=TAGGER) for r in RECS}
LOC_COVER = {sp: (sum(FEATS[r["paragraph_id"]]["loc_cover"][0] for r in BY_SPLIT[sp]), sum(FEATS[r["paragraph_id"]]["loc_cover"][1] for r in BY_SPLIT[sp]))
             for sp in SPLITS}
print("Đầu vị trí: tỉ lệ vị trí vàng có trong ứng viên cụm danh từ (gồm '?') =",
      {sp: f"{h}/{n}={h/max(n,1):.1%}" for sp, (h, n) in LOC_COVER.items()}, "| tách cụm bằng", "NLTK" if TAGGER else "cửa sổ từ (lùi)")
FEATS_RAND = {pid: dict(f, logE=torch.tensor(LOGE_RAND[pid], dtype=torch.float)) for pid, f in FEATS.items()}
def feats_for(variant, recs):
    src = FEATS_RAND if variant == "CRF_RANDE" else FEATS
    return [src[r["paragraph_id"]] for r in recs]
def model_variant(v):          # CRF_RANDE dùng kiến trúc CRF_FULL với log E xáo trộn
    return "CRF_FULL" if v == "CRF_RANDE" else v
print(f"featurize {len(FEATS)} đoạn trong {time.time()-t0:.1f}s")
def evaluate_fn(preds, recs):
    return pm.evaluate(preds, recs)

# Biến thể NONCE (đổi tên thực thể bằng từ vô nghĩa, nhãn giữ nguyên): dev chấm công khai, test lưu kín
_raws = {sp: pc.load_jsonl(str(RAW[f"grids.v1.{sp}.json"])) for sp in ("dev", "test")}
_keep = {"dev": {r["paragraph_id"] for r in DEV}, "test": {r["paragraph_id"] for r in TEST}}
NONCE = pr.nonce_records({sp: [x for x in _raws[sp] if str(x["para_id"]) in _keep[sp]] for sp in _raws}, TOPICS)
_e_ok = PP3.get("status") == "ok"
FEATS_NONCE = {sp: [pp.featurize(tok, r, p3.logE_matrix(E_MODEL, embed_words, r, device=DEVICE) if _e_ok else None, tagger=TAGGER) for r in NONCE[sp]]
               for sp in NONCE}
print("NONCE:", {sp: len(v) for sp, v in NONCE.items()}, "| ví dụ:", NONCE["dev"][0]["entities"], "|", NONCE["dev"][0]["sentences"][0])

## 8. Huấn luyện + chọn epoch trên DEV (5 biến thể × 3 seed); test dự đoán KÍN

In [ ]:
RESULTS = json.loads((STATE / "results.json").read_text()) if (STATE / "results.json").exists() else {}
PRED_DIR = OUT / "predictions"; PRED_DIR.mkdir(exist_ok=True)
CKPT_DIR = OUT / "checkpoints"; CKPT_DIR.mkdir(exist_ok=True)
SEALED_DIR = OUT / "sealed_test_predictions"; SEALED_DIR.mkdir(exist_ok=True)

def dump_preds(path, preds):
    with open(path, "w", encoding="utf-8") as f:
        for pid, p in preds.items():
            f.write(json.dumps(p, ensure_ascii=False) + "\n")

for v in VARIANTS:
    for seed in SEEDS:
        key = f"DONE|{v}|{seed}"
        if key in RESULTS:
            print("bỏ qua (đã có):", key); continue
        if tdone() > MAX_HOURS:
            print("[DỪNG] vượt MAX_HOURS"); break
        CK = STATE / f"inprogress_{v}_seed{seed}.pt"          # checkpoint theo epoch → bị ngắt thì chạy lại sẽ tiếp tục
        model, hist, bests = pp.train_variant(make_encoder, tok, model_variant(v), feats_for(v, TRAIN), DEV, feats_for(v, DEV),
                                              evaluate_fn, seed=seed, epochs=EPOCHS, device=DEVICE, ckpt_path=str(CK),
                                              patience=PATIENCE, stop_after_epochs=STOP_AFTER)
        for nm, (state, best_ep) in bests.items():
            name = nm if v == "M0" else v
            model.load_state_dict(state)
            dev_preds = pp.predict_all(model, DEV, feats_for(v, DEV), DEVICE, constrained=(nm == "M0c"))
            test_preds = pp.predict_all(model, TEST, feats_for(v, TEST), DEVICE, constrained=(nm == "M0c"))
            dump_preds(PRED_DIR / f"dev_{name}_seed{seed}.jsonl", dev_preds)
            dump_preds(SEALED_DIR / f"test_{name}_seed{seed}.jsonl", test_preds)      # KHÔNG chấm ở đây
            res = pm.evaluate(dev_preds, DEV)
            RESULTS[f"{name}|{seed}"] = {"variant": name, "seed": seed, "best_epoch": best_ep, "history": hist,
                                         "dev": pm.summary_row(name, res), "_per_paragraph": res["_per_paragraph"]}
            if v != "CRF_RANDE":
                c = (nm == "M0c")
                dn = pp.predict_all(model, NONCE["dev"], FEATS_NONCE["dev"], DEVICE, constrained=c)
                RESULTS[f"{name}|{seed}"]["dev_nonce"] = pm.summary_row(name, pm.evaluate(dn, NONCE["dev"]))
                dump_preds(SEALED_DIR / f"test_nonce_{name}_seed{seed}.jsonl",
                           pp.predict_all(model, NONCE["test"], FEATS_NONCE["test"], DEVICE, constrained=c))
            print(f"==> {name} seed={seed}: best_ep={best_ep} dev PAIR_F1={res['PAIR@all']['F1']:.3f} @any_imp={res['PAIR@any_implicit']['F1']:.3f}")
            if seed == SEEDS[0]:          # checkpoint TỐT NHẤT (theo dev) của mọi hệ thống, seed đầu, fp16 ~250 MB/file
                torch.save({k: t.half() if t.is_floating_point() else t for k, t in state.items()}, CKPT_DIR / f"{name}_seed{seed}_best.pt")
        RESULTS[key] = {"done": True, "epochs_run": len(hist), "early_stopped": len(hist) < EPOCHS}
        (STATE / "results.json").write_text(json.dumps(RESULTS, default=str))
        pp.cleanup_ckpt(str(CK))        # kết quả đã lưu → xoá checkpoint đang-chạy + bản tốt nhất tạm (tránh phình Output)
        del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None

## 9. Bảng ablation trên DEV (trung bình ± độ lệch chuẩn qua seed) + kiểm định cặp

In [ ]:
SYS = ["M0", "M0c", "CRF_NOPAIR", "CRF_NOE", "CRF_RANDE", "CRF_FULL"]
LABEL = {"M0": "M0 độc lập", "M0c": "M0c = M0 + luật", "CRF_NOPAIR": "M3-no-pair (CRF)", "CRF_NOE": "M3-no-E",
         "CRF_RANDE": "M2r (E xáo trộn)", "CRF_FULL": "M3 (đầy đủ)"}
METRICS = ["PAIR_F1", "PAIR_F1@src_imp", "PAIR_F1@tgt_imp", "PAIR_F1@any_imp", "IMP_ACC_DESTROY", "IMP_ACC_CREATE", "F1_MOVE", "EXIST_ACC",
           "VIOL_per_entity", "PAIR_LOC_F1"]
ABL = []
for s in SYS:
    rows = [RESULTS[f"{s}|{seed}"] for seed in SEEDS if f"{s}|{seed}" in RESULTS]
    if not rows:
        continue
    r = {"system": LABEL[s], "code": s, "n_seeds": len(rows), "best_epochs": [x["best_epoch"] for x in rows]}
    for m in METRICS:
        vals = [x["dev"][m] for x in rows if x["dev"][m] == x["dev"][m]]
        r[m] = statistics.mean(vals) if vals else float("nan")
        r[m + "_sd"] = statistics.stdev(vals) if len(vals) > 1 else 0.0
    ABL.append(r)
print(f"{'hệ thống':22s} " + " ".join(f"{m[:15]:>15s}" for m in METRICS))
for r in ABL:
    print(f"{r['system']:22s} " + " ".join(f"{r[m]:.3f}±{r[m+'_sd']:.3f}"[:15].rjust(15) for m in METRICS))

print("Khái quát (NONCE, dev): PAIR_F1 gốc → đổi tên (TB qua seed)")
NONCE_TAB = {}
for s_ in SYS:
    rows = [RESULTS[f"{s_}|{seed}"] for seed in SEEDS if f"{s_}|{seed}" in RESULTS and "dev_nonce" in RESULTS[f"{s_}|{seed}"]]
    if rows:
        a_ = statistics.mean(x["dev"]["PAIR_F1"] for x in rows); b_ = statistics.mean(x["dev_nonce"]["PAIR_F1"] for x in rows)
        ai = statistics.mean(x["dev"]["PAIR_F1@any_imp"] for x in rows); bi = statistics.mean(x["dev_nonce"]["PAIR_F1@any_imp"] for x in rows)
        NONCE_TAB[s_] = {"orig": a_, "nonce": b_, "orig_imp": ai, "nonce_imp": bi}
        print(f"  {LABEL[s_]:22s} {a_:.3f} → {b_:.3f}  | @any_imp {ai:.3f} → {bi:.3f}")

# Kiểm định cặp trên dev, gộp seed bằng cách so seed tương ứng; báo cáo seed đầu + trung bình Δ
PAIRS_TO_TEST = [("CRF_FULL", "M0c"), ("CRF_FULL", "CRF_NOE"), ("CRF_FULL", "CRF_RANDE"), ("CRF_NOE", "CRF_NOPAIR"),
                 ("CRF_NOPAIR", "M0c"), ("M0c", "M0")]
BOOT_DEV = {}
for a, b in PAIRS_TO_TEST:
    for sl in ("all", "any_implicit"):
        ds = []
        for seed in SEEDS:
            ka, kb = f"{a}|{seed}", f"{b}|{seed}"
            if ka in RESULTS and kb in RESULTS:
                ds.append(pm.paired_bootstrap_pair_f1({"_per_paragraph": RESULTS[ka]["_per_paragraph"]},
                                                      {"_per_paragraph": RESULTS[kb]["_per_paragraph"]}, sl, n_boot=2000, seed=SEED))
        if ds:
            BOOT_DEV[f"{a}-{b}@{sl}"] = {"delta_mean": statistics.mean(d["delta"] for d in ds),
                                         "seed0_ci": [ds[0]["ci_low"], ds[0]["ci_high"]], "n_seeds": len(ds)}
            x = BOOT_DEV[f"{a}-{b}@{sl}"]
            print(f"Δ {a} − {b} @{sl}: TB {x['delta_mean']:+.3f} (seed0 CI95 [{x['seed0_ci'][0]:+.3f}, {x['seed0_ci'][1]:+.3f}])")

## 10. PP2 theo đúng công thức (11): lọc cặp bằng XÁC SUẤT tồn tại dự đoán, khảo sát ngưỡng τ1, τ2 trên DEV (H1)

In [ ]:
# C_t = {(X,Y): p(e_{X,t-1}=1) > τ1, p(e_{Y,t-1}=1) < τ2}; p lấy từ biên CRF (forward-backward) của mô hình đã huấn luyện.
TAU1 = [0.01, 0.05, 0.1, 0.2, 0.3, 0.5]
TAU2 = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]
PP2_RES, PP2_GRID = {}, {}
for s_ in ["M0c", "CRF_FULL" if (CKPT_DIR / f"CRF_FULL_seed{SEEDS[0]}_best.pt").exists() else "CRF_NOE"]:
    ck = CKPT_DIR / f"{s_}_seed{SEEDS[0]}_best.pt"
    if not ck.exists():
        continue
    enc, hid = make_encoder()
    mdl = pp.PP4Model(enc, hid, "M0" if s_ == "M0c" else model_variant(s_)).to(DEVICE)
    mdl.load_state_dict({k: v.float() if v.is_floating_point() else v for k, v in torch.load(ck, map_location="cpu").items()})
    mdl.eval()
    probs = {r["paragraph_id"]: mdl.existence_probs(f, DEVICE).cpu() for r, f in zip(DEV, feats_for(s_, DEV))}
    grid = {}
    for t1 in TAU1:
        for t2 in TAU2:
            kept = total = gold = gold_kept = 0
            for r in DEV:
                P_ = probs[r["paragraph_id"]]
                n, T = len(r["entities"]), len(r["sentences"])
                G = {(g["sentence_id"], g["source_idx"], g["target_idx"]) for g in pc.gold_pairs_report(r)}
                for t in range(1, T + 1):
                    for x in range(n):
                        for y in range(n):
                            if x == y: continue
                            total += 1
                            ok = bool(P_[t - 1, x] > t1 and P_[t - 1, y] < t2)
                            kept += ok
                            if (t, x, y) in G:
                                gold += 1; gold_kept += ok
            grid[(t1, t2)] = {"candidate_ratio": kept / max(total, 1), "gold_recall": gold_kept / max(gold, 1)}
    PP2_GRID[s_] = {f"{a}|{b}": v for (a, b), v in grid.items()}
    # điểm vận hành khoá trước: τ1 = 0.1, τ2 = 0.9 ('đặt lỏng để hạn chế bỏ sót'); H1 đạt nếu phủ ≥ 95% và giữ ≤ 50%
    op = grid[(0.1, 0.9)]
    PP2_RES[s_] = dict(op, H1_pass=op["gold_recall"] >= 0.95 and op["candidate_ratio"] <= 0.5, n_gold=gold)
    print(f"PP2 [{s_}] τ1=0.1, τ2=0.9: giữ {op['candidate_ratio']:.1%} ứng viên, phủ {op['gold_recall']:.1%} cặp vàng → H1 {'ĐẠT' if PP2_RES[s_]['H1_pass'] else 'CHƯA ĐẠT'}")
    del mdl; torch.cuda.empty_cache() if DEVICE == "cuda" else None
if PP2_GRID:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(6, 4), dpi=120)
    for s_, col in zip(PP2_GRID, ["#2a78d6", "#6250d6"]):
        pts = sorted((v["candidate_ratio"], v["gold_recall"]) for v in PP2_GRID[s_].values())
        ax.plot([p[0] for p in pts], [p[1] for p in pts], "o", color=col, ms=5, label=s_, alpha=0.8)
    ax.axhline(0.95, color="#52514e", lw=1, ls="--"); ax.axvline(0.5, color="#52514e", lw=1, ls="--")
    ax.set_xlabel("tỉ lệ ứng viên còn lại"); ax.set_ylabel("độ phủ cặp vàng (dev)")
    ax.set_title("PP2: đánh đổi số cặp phải xét ↔ độ phủ (lưới τ1, τ2)", loc="left", fontsize=10); ax.legend(frameon=False)
    for sp_ in ("top", "right"): ax.spines[sp_].set_visible(False)
    (OUT / "figures").mkdir(exist_ok=True); fig.tight_layout(); fig.savefig(OUT / "figures" / "pp2_tradeoff.png"); plt.show(); plt.close(fig)

## 11. CV 5 phần theo chủ đề: M3 vs M0c trên toàn bộ 488 đoạn (kiểm định chính cho lát ngầm)

In [ ]:
CV = json.loads((STATE / "cv.json").read_text()) if (STATE / "cv.json").exists() else {}
CV_EPOCHS = {}
for v in CV_VARIANTS:
    names = ["M0c"] if v == "M0" else [v]
    eps = [RESULTS[f"{names[0]}|{s}"]["best_epoch"] for s in SEEDS if f"{names[0]}|{s}" in RESULTS]
    CV_EPOCHS[v] = int(statistics.median(eps)) if eps else EPOCHS
print("Số epoch cố định cho CV (trung vị epoch tốt nhất trên dev):", CV_EPOCHS)
CV_RECS = [r for r in RECS if not LIMIT_TRAIN or r in TRAIN + DEV + TEST]
if RUN_CV:
    for v in CV_VARIANTS:
        for fold in range(5):
            key = f"DONE|{v}|fold{fold}"
            if key in CV or tdone() > MAX_HOURS:
                continue
            tr = [r for r in CV_RECS if FOLDS[r["paragraph_id"]] != fold]
            te = [r for r in CV_RECS if FOLDS[r["paragraph_id"]] == fold]
            if LIMIT_TRAIN:
                tr = tr[:LIMIT_TRAIN]
            # không chọn epoch trên fold kiểm tra: huấn luyện đúng CV_EPOCHS[v] epoch, lấy mô hình cuối
            CK = STATE / f"inprogress_cv_{v}_fold{fold}.pt"
            model, hist, _ = pp.train_variant(make_encoder, tok, model_variant(v), feats_for(v, tr), te[:1], feats_for(v, te[:1]),
                                              evaluate_fn, seed=SEEDS[0], epochs=CV_EPOCHS[v], device=DEVICE, ckpt_path=str(CK),
                                              stop_after_epochs=STOP_AFTER)
            for nm in (["M0", "M0c"] if v == "M0" else [v]):
                preds = pp.predict_all(model, te, feats_for(v, te), DEVICE, constrained=(nm == "M0c"))
                CV[f"{nm}|fold{fold}"] = preds
            CV[key] = {"done": True}
            (STATE / "cv.json").write_text(json.dumps(CV))
            pp.cleanup_ckpt(str(CK))
            print(f"CV {v} fold {fold}: xong ({len(te)} đoạn)")
            del model; torch.cuda.empty_cache() if DEVICE == "cuda" else None
CV_RES, CV_BOOT = {}, {}
for nm in ["M0", "M0c", "CRF_FULL" if "CRF_FULL" in CV_VARIANTS else "CRF_NOE"]:
    allp = {}
    for fold in range(5):
        allp.update(CV.get(f"{nm}|fold{fold}", {}))
    recs_cov = [r for r in CV_RECS if r["paragraph_id"] in allp]
    if len(recs_cov) == len(CV_RECS):
        CV_RES[nm] = pm.evaluate(allp, CV_RECS)
if len(CV_RES) >= 2:
    m3 = "CRF_FULL" if "CRF_FULL" in CV_RES else "CRF_NOE"
    for sl in ("all", "src_implicit", "tgt_implicit", "any_implicit"):
        CV_BOOT[f"{m3}-M0c@{sl}"] = pm.paired_bootstrap_pair_f1(CV_RES[m3], CV_RES["M0c"], sl, n_boot=N_BOOT, seed=SEED)
    for nm, res in CV_RES.items():
        print(f"CV {nm:10s} PAIR_F1={res['PAIR@all']['F1']:.3f} @src_imp={res['PAIR@src_implicit']['F1']:.3f} "
              f"@tgt_imp={res['PAIR@tgt_implicit']['F1']:.3f} @any_imp={res['PAIR@any_implicit']['F1']:.3f} "
              f"IMP_D={res['IMPLICIT_ACC']['DESTROY']['acc']:.3f} IMP_C={res['IMPLICIT_ACC']['CREATE']['acc']:.3f} VIOL={res['VIOLATION_RATE_per_entity']:.3f}")
    for k, b in CV_BOOT.items():
        print(f"Δ CV {k}: {b['delta']:+.3f} CI95 [{b['ci_low']:+.3f}, {b['ci_high']:+.3f}] (n={b['n_paragraphs']} đoạn)")
else:
    print("CV chưa đủ (RUN_CV tắt hoặc vượt MAX_HOURS).")

## 12. Biểu đồ: loss train/dev, F1 dev theo epoch, ablation

In [ ]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
FIG = OUT / "figures"; FIG.mkdir(exist_ok=True)
COL = {"M0": "#2a78d6", "CRF_NOPAIR": "#eb6834", "CRF_NOE": "#1baf7a", "CRF_RANDE": "#eda100", "CRF_FULL": "#6250d6"}
def _style(ax):
    for s in ("top", "right"): ax.spines[s].set_visible(False)
    ax.grid(True, color="#e6e5e0", linewidth=0.8); ax.set_axisbelow(True)
fig, axes = plt.subplots(1, 3, figsize=(15, 4), dpi=120)
for v in VARIANTS:
    k = f"{'M0' if v == 'M0' else v}|{SEEDS[0]}"
    if k not in RESULTS: continue
    h = RESULTS[k]["history"]; ep = [x["epoch"] for x in h]
    axes[0].plot(ep, [x["train_loss"] for x in h], color=COL[v], lw=2, label=v)
    axes[1].plot(ep, [x["dev_loss"] for x in h], color=COL[v], lw=2, label=v)
    f1k = "M0c:dev_PAIR_F1" if v == "M0" else f"{model_variant(v)}:dev_PAIR_F1"
    axes[2].plot(ep, [x[f1k] for x in h], color=COL[v], lw=2, label=("M0c" if v == "M0" else v))
for ax, t in zip(axes, ["Loss train (theo ô)", "Loss dev (theo ô)", "PAIR_F1 dev"]):
    ax.set_title(t, loc="left", fontsize=10); ax.set_xlabel("epoch"); _style(ax)
axes[2].legend(frameon=False, fontsize=8)
fig.suptitle(f"Huấn luyện PP4, seed {SEEDS[0]} (M0: CE; CRF: −log p(y_vàng)/số ô, nên không so trực tiếp độ lớn loss giữa M0 và CRF)", fontsize=9, x=0.01, ha="left")
fig.tight_layout(); fig.savefig(FIG / "pp4_training_curves.png"); plt.show(); plt.close(fig)

if ABL:
    fig, ax = plt.subplots(figsize=(10, 4), dpi=120)
    ms = ["PAIR_F1", "PAIR_F1@src_imp", "PAIR_F1@tgt_imp", "PAIR_F1@any_imp"]
    w = 0.8 / len(ABL)
    for k, r in enumerate(ABL):
        xs = [j + k * w for j in range(len(ms))]
        ax.bar(xs, [r[m] for m in ms], width=w * 0.92, yerr=[r[m + "_sd"] for m in ms], capsize=2,
               color=["#2a78d6", "#8fb8ea", "#eb6834", "#1baf7a", "#eda100", "#6250d6"][k], label=r["system"])
    ax.set_xticks([j + 0.4 - w / 2 for j in range(len(ms))], ["PAIR_F1", "@nguồn ngầm", "@đích ngầm", "@có phía ngầm"])
    ax.set_title("Ablation trên dev (TB ± SD qua seed)", loc="left", fontsize=10); ax.legend(frameon=False, fontsize=8, ncol=3); _style(ax)
    fig.tight_layout(); fig.savefig(FIG / "pp4_ablation_dev.png"); plt.show(); plt.close(fig)
if PP3.get("history"):
    fig, ax = plt.subplots(figsize=(6, 3.5), dpi=120)
    h = PP3["history"]
    ax.plot([x["epoch"] for x in h], [x["train_loss"] for x in h], color="#2a78d6", lw=2, label="train")
    ax.plot([x["epoch"] for x in h], [x["val_loss"] for x in h], color="#eb6834", lw=2, label="validation")
    ax.set_title("PP3: BCE của E(X,Y)", loc="left", fontsize=10); ax.legend(frameon=False); _style(ax)
    fig.tight_layout(); fig.savefig(FIG / "pp3_loss.png"); plt.show(); plt.close(fig)

## 13. Lưu đầu ra + gửi lại

In [ ]:
def jdump(obj, path):
    Path(path).write_text(json.dumps(obj, indent=1, ensure_ascii=False, default=str), encoding="utf-8")
CV_SUMMARY = {nm: pm.summary_row(nm, r) for nm, r in CV_RES.items()}
SUMMARY = {"run_tag": RUN_TAG, "env": ENV, "encoder": f"{ENCODER_ID}@{ENCODER_REV}", "variants": VARIANTS, "seeds": SEEDS, "epochs": EPOCHS,
           "pp3": {k: v for k, v in PP3.items() if k != "history"}, "pp3_history": PP3.get("history"),
           "ablation_dev": ABL, "nonce_dev": NONCE_TAB, "bootstrap_dev": BOOT_DEV, "pp2_predicted": PP2_RES, "pp2_grid": PP2_GRID, "location_candidate_cover": LOC_COVER, "cv_epochs": CV_EPOCHS, "cv": CV_SUMMARY,
           "cv_bootstrap": CV_BOOT, "histories": {k: v["history"] for k, v in RESULTS.items() if "history" in v and k.endswith(f"|{SEEDS[0]}")},
           "sealed_test_files": sorted(p.name for p in SEALED_DIR.glob("*.jsonl")), "runtime_h": round(tdone(), 2),
           "restored_files": len(RESTORED), "early_stopping": {k: v for k, v in RESULTS.items() if k.startswith("DONE|")},
           "checkpoints": sorted(p.name for p in CKPT_DIR.glob("*.pt")),
           "versions": {"core": pc.CORE_VERSION, "metrics": pm.METRICS_VERSION, "pp3": p3.PP3_VERSION, "pp4": pp.PP4_VERSION}}
jdump(SUMMARY, OUT / "summary_05.json")
if PP3.get("status") == "ok":
    jdump({"top_pairs": PAIR_CNT.most_common(5000), "coverage": {k: v for k, v in PP3.items() if k.startswith("coverage")}}, OUT / "pp3_pairs_top5000.json")
    jdump(LOGE, OUT / "pp3_logE_propara.json")
ZIP = WORK_ROOT / f"{RUN_TAG}.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and "code" not in p.parts and "raw" not in p.parts and p.suffix not in (".pt", ".tmp"):
            z.write(p, Path(RUN_TAG) / p.relative_to(OUT))
print("=== GỬI LẠI ===")
print(f"notebook=05_pp3_pp4 gpus={ENV['gpus']} runtime={SUMMARY['runtime_h']}h seeds={SEEDS} epochs<={EPOCHS} patience={PATIENCE} khoi_phuc={len(RESTORED)} file")
print("đã xong:", sorted(k[5:] for k in RESULTS if k.startswith("DONE|")), "| CV:", sorted(k[5:] for k in CV if k.startswith("DONE|")))
print("checkpoint tốt nhất:", SUMMARY["checkpoints"])
print("PP3:", PP3.get("status"), {k: PP3.get(k) for k in ("n_pairs_distinct", "n_pair_mentions", "coverage_all_min1", "propara_auc_dev", "best_val")})
for r in ABL:
    print(f"  DEV {r['system']:20s} PAIR_F1={r['PAIR_F1']:.3f}±{r['PAIR_F1_sd']:.3f} @any_imp={r['PAIR_F1@any_imp']:.3f}±{r['PAIR_F1@any_imp_sd']:.3f} LOC={r['PAIR_LOC_F1']:.3f} "
          f"IMP_D={r['IMP_ACC_DESTROY']:.3f} IMP_C={r['IMP_ACC_CREATE']:.3f} VIOL={r['VIOL_per_entity']:.3f} ep={r['best_epochs']}")
for k, x in BOOT_DEV.items():
    print(f"  Δdev {k}: {x['delta_mean']:+.3f} seed0CI {x['seed0_ci'][0]:+.3f}..{x['seed0_ci'][1]:+.3f}")
print("  NONCE dev:", {k: {kk: round(vv, 3) for kk, vv in v.items()} for k, v in NONCE_TAB.items()})
print("  PP2 (τ1=0.1, τ2=0.9):", PP2_RES, "| phủ vị trí ứng viên:", LOC_COVER)
for nm, r in CV_SUMMARY.items():
    print(f"  CV {nm:10s} PAIR_F1={r['PAIR_F1']:.3f} @src={r['PAIR_F1@src_imp']:.3f} @tgt={r['PAIR_F1@tgt_imp']:.3f} @any={r['PAIR_F1@any_imp']:.3f}")
for k, b in CV_BOOT.items():
    print(f"  ΔCV {k}: {b['delta']:+.3f} [{b['ci_low']:+.3f}, {b['ci_high']:+.3f}]")
print("=== HẾT ===")